# Connectivity 3 — Degeneracy analysis of GLM connectivity (Marder framework)

| | |
|---|---|
| **Original** | `Liam_work/tests/graphical_analysis/AD_Degeneracy_Analysis.ipynb` |
| **Status** | Side project (Feb 2026). |

Treats each animal's per-region connectivity summaries (strength / density per cell-type pair, E/I ratio)
like the channel-conductance measurements of Prinz 2004 / Schulz 2006 and asks whether AD breaks their
co-variation: pairwise correlations, PCA of the parameter space, cross-region coordination, Mahalanobis
distance from the WT "solution space", age-stratified analysis (212-day cutoff), plus leave-one-animal-out
outlier checks for every result. Headline (Liam's notes): a biphasic cross-region coordination trajectory —
young AD hyper-coordinated, old AD de-coordinated, WT flat.

Reads `data/glm_results/glm_connectivity_results_20animals_REM.pkl`.

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

In [ ]:
# =============================================================================
# LAB-WIDE PREAMBLE — CPU thread limits + figure-export font config
# (added in the repo copy; set BEFORE numpy/scipy/sklearn/xgboost are imported)
# =============================================================================
import os
import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family': 'DejaVu Sans'})

# Limit CPU threads — shared server "marvin"
os.environ['OMP_NUM_THREADS']        = '4'
os.environ['OPENBLAS_NUM_THREADS']   = '4'
os.environ['MKL_NUM_THREADS']        = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS']    = '4'

# Degeneracy Analysis: AD vs WT Functional Connectivity (REM Sleep)
## Do AD mice lose the correlated parameter structure that maintains network function?

**Motivation** (Marder lab framework):  
- Prinz et al. (2004): ~20% of 20M parameter combinations produce functional pyloric rhythm — the "solution space" is vast but structured  
- Schulz et al. (2006): Same neuron type shows 2-4x variability across animals, but *within* an animal, coupled neurons maintain correlated channel expression  
- Temporal et al. (2014): These correlations are *actively maintained* by activity-dependent feedback  
- Goaillard & Marder (2021): Disease/perturbation can break these correlations → variable, unpredictable responses  

**Hypothesis**: AD doesn't simply increase or decrease connectivity — it disrupts the *covariance structure* of connectivity parameters.  
WT animals vary in absolute coupling strengths but maintain tight relationships between connection types.  
AD animals lose this coordination, falling outside the "solution space" of functional parameter combinations.

**Data**: Point process GLM connectivity estimated during REM sleep (EBIC regularization, gamma=1.0).  
REM chosen over NREM to avoid UP/DOWN state confound inflating pairwise coupling.

**Analyses**:  
1. Within-animal parameter correlation structure  
2. Across-animal variability (coefficient of variation)  
3. Covariance structure comparison (AD vs WT)  
4. PCA of connectivity parameter space  
5. Solution space geometry and outlier detection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms
from scipy import stats
from scipy.spatial.distance import mahalanobis
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from collections import defaultdict, Counter
import pickle
import warnings
import os

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 8)

RESULTS_DIR = '/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/data/glm_results/'  # repo copy
RESULTS_FILE = os.path.join(RESULTS_DIR, 'glm_connectivity_results_20animals_REM.pkl')

with open(RESULTS_FILE, 'rb') as f:
    glm_data = pickle.load(f)

# Exclude animals with data quality issues
# ZBR113: only 38 neurons after rescoring (>2 SD below mean), only 1 of 4 regions
#         passes MIN_NEURONS=15 threshold (ACad=18; CA1=7, RSPv=9, V1=4 all fail)
# CAF104: spike sorting instability confirmed in spike classification analysis.
#         81% of neurons have within-epoch ISIs >100s (next closest: 9.7%).
#         Also flagged as MODERATE outlier (4 flags) in degeneracy outlier detection:
#         8 features with |Z|>2, shifts AD Mahalanobis outlier count, distorts Old WT
#         cross-region coordination (Δr=-0.165 when removed).
EXCLUDE_ANIMALS = ['ZBR113', 'CAF104']
for a in EXCLUDE_ANIMALS:
    if a in glm_data:
        print(f'  EXCLUDED {a}: {glm_data[a]["n_neurons"]} neurons')
        del glm_data[a]

print(f'Loaded {len(glm_data)} animals (REM sleep GLM, EBIC gamma=1.0)')
for a in sorted(glm_data.keys()):
    print(f'  {a}: {glm_data[a]["genotype"]}')

## 1. Extract Connectivity Parameters Per Animal

For each animal, extract a vector of connectivity "features" — analogous to the conductance
measurements in Schulz et al. These are the parameters that can vary across animals while
(potentially) maintaining network function.

Features per region:
- Mean coupling strength for each cell-type pair (RSU→RSU, RSU→FS, FS→RSU, FS→FS)
- Connection density for each cell-type pair
- E/I ratio (excitatory vs inhibitory connections)
- Mean excitatory and inhibitory coupling strengths

In [ ]:
PAIR_TYPES = ['RSU->RSU', 'RSU->FS', 'FS->RSU', 'FS->FS']

# Canonical region mapping (old probe names -> standard)
REGION_MAP = {
    'ACad': 'AC', 'ACC': 'AC',   # anterior cingulate variants
    'CA1': 'CA1',
    'RSPv': 'RSC', 'RSC': 'RSC', # retrosplenial variants  
    'V1': 'V1',
}
REGIONS = ['AC', 'CA1', 'RSC', 'V1']

# Minimum neuron count — regions with fewer neurons get unreliable
# connectivity estimates (too few directed pairs for meaningful density/strength).
# Set to 7: gives 42 directed pairs, and each neuron's GLM only fits 6 coupling
# terms — massively overdetermined with hours of recording data.
# EBIC gamma=1.0 already controls false positive connections conservatively.
MIN_NEURONS = 7


def extract_animal_params(animal_name, res):
    """Extract connectivity parameter vector for one animal.
    
    Returns dict with per-region, per-cell-type-pair metrics.
    Regions with < MIN_NEURONS neurons are excluded.
    """
    info = res['neuron_info']
    params = {'animal': animal_name, 'genotype': res['genotype']}
    skipped = []
    
    for reg_raw, rdata in res['regions'].items():
        reg = REGION_MAP.get(reg_raw, reg_raw)
        if reg not in REGIONS:
            continue
            
        conn = rdata['connectivity']
        idx = rdata['neuron_indices']
        n = len(idx)
        
        if n < MIN_NEURONS:
            skipped.append(f'{reg}(n={n})')
            continue
            
        # Collect weights by cell-type pair
        pair_weights = defaultdict(list)
        pair_all = defaultdict(int)  # total possible
        
        for si in range(n):
            for ti in range(n):
                if si == ti:
                    continue
                src_t = info[idx[si]]['cell_type']
                tgt_t = info[idx[ti]]['cell_type']
                pair = f'{src_t}->{tgt_t}'
                if pair in PAIR_TYPES:
                    pair_all[pair] += 1
                    w = conn[si, ti]
                    if abs(w) > 1e-6:
                        pair_weights[pair].append(w)
        
        for pt in PAIR_TYPES:
            ws = pair_weights.get(pt, [])
            total = pair_all.get(pt, 0)
            
            # Mean absolute coupling strength
            params[f'{reg}_{pt}_strength'] = np.mean(np.abs(ws)) if ws else 0.0
            # Connection density
            params[f'{reg}_{pt}_density'] = len(ws) / total if total > 0 else 0.0
            # Mean excitatory coupling
            exc = [w for w in ws if w > 0]
            params[f'{reg}_{pt}_exc'] = np.mean(exc) if exc else 0.0
            # Mean inhibitory coupling
            inh = [w for w in ws if w < 0]
            params[f'{reg}_{pt}_inh'] = np.mean(np.abs(inh)) if inh else 0.0
        
        # Region-level E/I ratio
        all_w = [w for ws in pair_weights.values() for w in ws]
        n_exc = sum(1 for w in all_w if w > 0)
        n_inh = sum(1 for w in all_w if w < 0)
        params[f'{reg}_ei_ratio'] = n_exc / max(n_inh, 1)
        
        # Region-level overall density
        mask = ~np.eye(n, dtype=bool)
        params[f'{reg}_density'] = np.mean(np.abs(conn[mask]) > 1e-6)
        params[f'{reg}_mean_strength'] = np.mean(np.abs(conn[mask][np.abs(conn[mask]) > 1e-6])) if np.any(np.abs(conn[mask]) > 1e-6) else 0.0
        params[f'{reg}_n_neurons'] = n
    
    if skipped:
        print(f'  {animal_name}: skipped {skipped} (< {MIN_NEURONS} neurons)')
    
    return params


# Extract for all animals
all_params = []
for animal in sorted(glm_data.keys()):
    p = extract_animal_params(animal, glm_data[animal])
    all_params.append(p)

# Get all parameter names (numeric features only)
# Use union of all keys across animals (some regions may be missing)
all_feature_keys = set()
for p in all_params:
    for k in p:
        if k not in {'animal', 'genotype'} and not k.endswith('_n_neurons'):
            all_feature_keys.add(k)
feature_names = sorted(all_feature_keys)

print(f'\nExtracted {len(feature_names)} features per animal (MIN_NEURONS={MIN_NEURONS})')
print(f'Features: {feature_names[:10]}...')

# Build feature matrix
animals = [p['animal'] for p in all_params]
genotypes = [p['genotype'] for p in all_params]
X = np.array([[p.get(f, 0.0) for f in feature_names] for p in all_params])

wt_mask = np.array([g == 'WT' for g in genotypes])
ad_mask = np.array([g == 'AD' for g in genotypes])

print(f'\nFeature matrix: {X.shape} ({wt_mask.sum()} WT, {ad_mask.sum()} AD)')
print(f'\nFeature summary (mean +/- std):')
for i, fn in enumerate(feature_names[:20]):
    wt_vals = X[wt_mask, i]
    ad_vals = X[ad_mask, i]
    print(f'  {fn:35s}: WT={np.mean(wt_vals):.4f}+/-{np.std(wt_vals):.4f}  '
          f'AD={np.mean(ad_vals):.4f}+/-{np.std(ad_vals):.4f}')

In [ ]:
# =============================================================================
# OUTLIER DETECTION 1: Per-Animal Feature Z-Scores
# =============================================================================
# For each connectivity feature, compute Z-score per animal within its genotype.
# Animals with many extreme Z-scores (|Z| > 2) across features are potential
# outliers that could drive group differences.
# =============================================================================

print('=' * 80)
print('OUTLIER DETECTION: Per-Animal Feature Z-Scores (within genotype)')
print('=' * 80)

# Initialize outlier tracking dict (will accumulate flags across all analyses)
outlier_flags = {a: [] for a in animals}

# Compute Z-scores within genotype for each feature
animal_zscores = np.zeros((len(animals), len(feature_names)))

for j, fn in enumerate(feature_names):
    for mask, label in [(wt_mask, 'WT'), (ad_mask, 'AD')]:
        vals = X[mask, j]
        m, s = np.mean(vals), np.std(vals)
        if s > 1e-8:
            animal_zscores[mask, j] = (vals - m) / s
        else:
            animal_zscores[mask, j] = 0

# Count extreme Z-scores per animal
n_extreme = np.sum(np.abs(animal_zscores) > 2, axis=1)
n_very_extreme = np.sum(np.abs(animal_zscores) > 3, axis=1)
max_abs_z = np.max(np.abs(animal_zscores), axis=1)

# Flag animals
for i, a in enumerate(animals):
    if n_extreme[i] >= 5:
        outlier_flags[a].append(f'EXTREME_Z: {n_extreme[i]} features with |Z|>2')
    if n_very_extreme[i] >= 2:
        outlier_flags[a].append(f'VERY_EXTREME_Z: {n_very_extreme[i]} features with |Z|>3')

# Print sorted by number of extreme features
print(f'\n{"Animal":<10s} {"Geno":>4s} {"Age":>6s} {"#|Z|>2":>8s} {"#|Z|>3":>8s} '
      f'{"MaxZ":>8s} {"Top extreme feature":>40s}')
print('-' * 95)
sort_idx = np.argsort(-n_extreme)
for i in sort_idx:
    a = animals[i]
    age = ANIMAL_AGES.get(a, 0) if 'ANIMAL_AGES' in globals() else 0
    worst_feat_idx = np.argmax(np.abs(animal_zscores[i]))
    worst_feat = feature_names[worst_feat_idx]
    worst_z = animal_zscores[i, worst_feat_idx]
    flag = ' *** OUTLIER' if n_extreme[i] >= 5 else ''
    print(f'{a:<10s} {genotypes[i]:>4s} {age:>5.0f}d {n_extreme[i]:>8d} '
          f'{n_very_extreme[i]:>8d} {max_abs_z[i]:>8.2f} {worst_feat:>35s} (Z={worst_z:+.2f}){flag}')

# Heatmap of Z-scores for top features (those with any |Z| > 2)
extreme_feats = [j for j in range(len(feature_names)) 
                 if np.any(np.abs(animal_zscores[:, j]) > 2)]
if extreme_feats:
    fig, ax = plt.subplots(figsize=(max(10, len(extreme_feats)*0.5), 8))
    im = ax.imshow(animal_zscores[:, extreme_feats], cmap='RdBu_r', vmin=-3, vmax=3,
                   aspect='auto')
    ax.set_xticks(range(len(extreme_feats)))
    feat_labels = [feature_names[j].replace('_strength', '_str')
                   .replace('RSU->RSU', 'R→R').replace('RSU->FS', 'R→F')
                   .replace('FS->RSU', 'F→R').replace('FS->FS', 'F→F')
                   for j in extreme_feats]
    ax.set_xticklabels(feat_labels, rotation=90, fontsize=7)
    ax.set_yticks(range(len(animals)))
    ylabels = [f'{a} ({genotypes[i]})' for i, a in enumerate(animals)]
    ax.set_yticklabels(ylabels, fontsize=8)
    plt.colorbar(im, ax=ax, label='Z-score (within genotype)')
    ax.set_title('Per-Animal Feature Z-Scores\n(Red = high, Blue = low, within genotype)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print(f'\nTotal features: {len(feature_names)}')
print(f'Features with any |Z| > 2: {len(extreme_feats)}')

## 4. Key Pairwise Correlations (Schulz Analog)

Schulz et al. found specific correlations between channel types (e.g., Shal & BK-KCa mRNA  
correlated in LP neurons). Here we look at the most biologically meaningful pairwise  
relationships:

- **RSU→FS vs FS→RSU** (reciprocal E/I balance — should co-vary to maintain stability)  
- **Same pair type across regions** (e.g., RSU→RSU in AC vs V1 — do animals scale globally?)  
- **Excitatory vs inhibitory strength within region** (compensatory like Shal vs BK-KCa)

In [ ]:
def scatter_with_regression(ax, x_wt, y_wt, x_ad, y_ad, xlabel, ylabel, title):
    """Scatter plot with separate regression lines for WT and AD."""
    # WT
    ax.scatter(x_wt, y_wt, c='#2196F3', s=80, alpha=0.8, label='WT', edgecolors='black', linewidth=0.5)
    if len(x_wt) > 2 and np.std(x_wt) > 1e-8 and np.std(y_wt) > 1e-8:
        r_wt, p_wt = stats.pearsonr(x_wt, y_wt)
        slope, intercept = np.polyfit(x_wt, y_wt, 1)
        xrange = np.linspace(min(x_wt), max(x_wt), 50)
        ax.plot(xrange, slope * xrange + intercept, '#2196F3', ls='--', lw=2)
        ax.text(0.02, 0.98, f'WT: r={r_wt:.2f}, p={p_wt:.3f}',
                transform=ax.transAxes, va='top', fontsize=8, color='#2196F3')
    
    # AD
    ax.scatter(x_ad, y_ad, c='#F44336', s=80, alpha=0.8, label='AD', edgecolors='black', linewidth=0.5)
    if len(x_ad) > 2 and np.std(x_ad) > 1e-8 and np.std(y_ad) > 1e-8:
        r_ad, p_ad = stats.pearsonr(x_ad, y_ad)
        slope, intercept = np.polyfit(x_ad, y_ad, 1)
        xrange = np.linspace(min(x_ad), max(x_ad), 50)
        ax.plot(xrange, slope * xrange + intercept, '#F44336', ls='--', lw=2)
        ax.text(0.02, 0.90, f'AD: r={r_ad:.2f}, p={p_ad:.3f}',
                transform=ax.transAxes, va='top', fontsize=8, color='#F44336')
    
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)


def get_feature(name):
    """Get feature vector by name."""
    idx = feature_names.index(name)
    return X[wt_mask, idx], X[ad_mask, idx]


# Key biologically motivated pairwise comparisons
comparisons = []

# 1. Reciprocal E/I balance within each region
for reg in REGIONS:
    f1 = f'{reg}_RSU->FS_strength'
    f2 = f'{reg}_FS->RSU_strength'
    if f1 in feature_names and f2 in feature_names:
        comparisons.append((f1, f2, f'Reciprocal balance: {reg}'))

# 2. Same pair type across regions (global scaling)
for pt in PAIR_TYPES:
    reg_pairs = [(r1, r2) for i, r1 in enumerate(REGIONS) 
                 for r2 in REGIONS[i+1:]
                 if f'{r1}_{pt}_strength' in feature_names 
                 and f'{r2}_{pt}_strength' in feature_names]
    for r1, r2 in reg_pairs[:1]:  # just first pair to limit plots
        comparisons.append((f'{r1}_{pt}_strength', f'{r2}_{pt}_strength', 
                           f'{pt} across regions: {r1} vs {r2}'))

# 3. Excitatory vs inhibitory within region
for reg in REGIONS:
    f1 = f'{reg}_RSU->RSU_exc'
    f2 = f'{reg}_RSU->RSU_inh'
    if f1 in feature_names and f2 in feature_names:
        comparisons.append((f1, f2, f'E vs I (RSU→RSU): {reg}'))

n_plots = len(comparisons)
ncols = 4
nrows = (n_plots + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
axes = axes.flat

for i, (f1, f2, title) in enumerate(comparisons):
    wt_x, ad_x = get_feature(f1)
    wt_y, ad_y = get_feature(f2)
    scatter_with_regression(axes[i], wt_x, wt_y, ad_x, ad_y,
                           f1.split('_', 1)[1], f2.split('_', 1)[1], title)

for i in range(n_plots, nrows * ncols):
    axes[i].set_visible(False)

plt.suptitle('Pairwise Parameter Correlations (Schulz et al. Analog)\n'
             'Tight WT correlations + broken AD correlations = degeneracy disruption',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# OUTLIER DETECTION 2: Pairwise Correlation Influence (Leave-One-Out)
# =============================================================================
# For each pairwise scatter (Schulz analog), remove each animal one at a time
# and see how much the WT and AD correlations change. Animals whose removal
# causes large correlation swings are driving the result.
# =============================================================================

print('=' * 80)
print('OUTLIER DETECTION: Pairwise Correlation — Leave-One-Out Influence')
print('=' * 80)

wt_indices = np.where(wt_mask)[0]
ad_indices = np.where(ad_mask)[0]

influence_scores = {a: [] for a in animals}

for f1, f2, comp_title in comparisons:
    if f1 not in feature_names or f2 not in feature_names:
        continue
    j1 = feature_names.index(f1)
    j2 = feature_names.index(f2)
    
    for geno_label, indices in [('WT', wt_indices), ('AD', ad_indices)]:
        x_all = X[indices, j1]
        y_all = X[indices, j2]
        if len(x_all) < 4 or np.std(x_all) < 1e-8 or np.std(y_all) < 1e-8:
            continue
        r_full, _ = stats.pearsonr(x_all, y_all)
        
        for k, idx in enumerate(indices):
            # Leave this animal out
            keep = np.ones(len(indices), dtype=bool)
            keep[k] = False
            x_loo = x_all[keep]
            y_loo = y_all[keep]
            if len(x_loo) < 3 or np.std(x_loo) < 1e-8 or np.std(y_loo) < 1e-8:
                continue
            r_loo, _ = stats.pearsonr(x_loo, y_loo)
            delta_r = r_loo - r_full  # positive = removing improves correlation
            influence_scores[animals[idx]].append({
                'comparison': comp_title, 'genotype': geno_label,
                'r_full': r_full, 'r_loo': r_loo, 'delta_r': delta_r
            })

# Summarize: mean absolute influence per animal
print(f'\n{"Animal":<10s} {"Geno":>4s} {"Mean|ΔR|":>10s} {"Max|ΔR|":>10s} '
      f'{"#Large":>8s} {"Worst comparison":>40s}')
print('-' * 90)

animal_mean_influence = {}
for a in animals:
    scores = influence_scores[a]
    if not scores:
        continue
    abs_deltas = [abs(s['delta_r']) for s in scores]
    mean_inf = np.mean(abs_deltas)
    max_inf = np.max(abs_deltas)
    n_large = sum(1 for d in abs_deltas if d > 0.15)
    worst = max(scores, key=lambda s: abs(s['delta_r']))
    animal_mean_influence[a] = mean_inf
    
    flag = ' *** HIGH INFLUENCE' if mean_inf > 0.10 else ''
    if mean_inf > 0.10:
        outlier_flags[a].append(f'HIGH_CORR_INFLUENCE: mean|ΔR|={mean_inf:.3f}')
    
    print(f'{a:<10s} {worst["genotype"]:>4s} {mean_inf:>10.4f} {max_inf:>10.4f} '
          f'{n_large:>8d} {worst["comparison"]:>35s} (ΔR={worst["delta_r"]:+.3f}){flag}')

# Bar chart of influence scores
fig, ax = plt.subplots(figsize=(12, 5))
sorted_animals = sorted(animal_mean_influence.keys(), 
                        key=lambda a: animal_mean_influence[a], reverse=True)
colors = ['#F44336' if genotypes[animals.index(a)] == 'AD' else '#2196F3' 
          for a in sorted_animals]
bars = ax.bar(range(len(sorted_animals)), 
              [animal_mean_influence[a] for a in sorted_animals],
              color=colors, edgecolor='#333333', linewidth=0.5)
ax.set_xticks(range(len(sorted_animals)))
ax.set_xticklabels(sorted_animals, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Mean |ΔR| (correlation change when removed)', fontsize=11)
ax.set_title('Pairwise Correlation Influence Score\n(How much does each animal affect correlations?)',
             fontsize=13, fontweight='bold')
ax.axhline(0.10, color='red', ls='--', alpha=0.5, label='High influence threshold')
from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], color='#2196F3', lw=8, label='WT'),
    Line2D([0], [0], color='#F44336', lw=8, label='AD'),
    Line2D([0], [0], color='red', ls='--', label='Threshold')
], fontsize=9)
plt.tight_layout()
plt.show()

## 5. PCA of Connectivity Parameter Space

**Prinz et al. analog**: The solution space of "good" networks is a structured region in  
high-dimensional parameter space. PCA reveals:  
- **PC1** typically captures "homogeneous scaling" (all conductances up/down together)  
- **PC2+** capture "conductance ratios" — the functionally important variation  

If WT animals cluster in one region of parameter space (the "solution space") while  
AD animals scatter more widely or occupy a different region, this suggests AD pushes  
circuits outside functional parameter combinations.

In [ ]:
# Use strength + density features for PCA
pca_features = [f for f in feature_names 
                if ('_strength' in f or '_ei_ratio' in f) 
                and not f.endswith('_n_neurons')]

# Filter to features with variance
pca_feat_idx = []
pca_feat_names = []
for f in pca_features:
    idx = feature_names.index(f)
    if np.std(X[:, idx]) > 1e-8:
        pca_feat_idx.append(idx)
        pca_feat_names.append(f)

X_pca_raw = X[:, pca_feat_idx]

# Standardize (like Schulz et al. — normalize each feature)
scaler = StandardScaler()
X_pca = scaler.fit_transform(X_pca_raw)

# PCA
pca = PCA()
scores = pca.fit_transform(X_pca)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Panel A: Scree plot
ax = axes[0, 0]
var_explained = pca.explained_variance_ratio_
cumvar = np.cumsum(var_explained)
n_show = min(10, len(var_explained))
ax.bar(range(n_show), var_explained[:n_show] * 100, alpha=0.7, color='steelblue')
ax.plot(range(n_show), cumvar[:n_show] * 100, 'ro-', markersize=6)
ax.axhline(80, color='gray', ls='--', alpha=0.3)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Scree Plot\n(Red line = cumulative)')
n_80 = np.searchsorted(cumvar, 0.8) + 1
ax.text(0.7, 0.5, f'{n_80} PCs for 80%\nvariance', transform=ax.transAxes, fontsize=10)

# Panel B: PC1 vs PC2 — the solution space
ax = axes[0, 1]
ax.scatter(scores[wt_mask, 0], scores[wt_mask, 1], c='#2196F3', s=100, alpha=0.8,
           label='WT', edgecolors='black', linewidth=0.5, zorder=5)
ax.scatter(scores[ad_mask, 0], scores[ad_mask, 1], c='#F44336', s=100, alpha=0.8,
           label='AD', edgecolors='black', linewidth=0.5, zorder=5)

# Label each point
for i, a in enumerate(animals):
    ax.annotate(a, (scores[i, 0], scores[i, 1]), fontsize=6,
                xytext=(4, 4), textcoords='offset points')

# Draw 95% confidence ellipses
for mask, color, label in [(wt_mask, '#2196F3', 'WT'), (ad_mask, '#F44336', 'AD')]:
    if mask.sum() >= 3:
        mean_x = np.mean(scores[mask, 0])
        mean_y = np.mean(scores[mask, 1])
        cov = np.cov(scores[mask, 0], scores[mask, 1])
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        angle = np.degrees(np.arctan2(eigenvectors[1, 1], eigenvectors[0, 1]))
        # 95% chi2 with 2 dof = 5.991
        width = 2 * np.sqrt(5.991 * eigenvalues[1])
        height = 2 * np.sqrt(5.991 * eigenvalues[0])
        ell = Ellipse(xy=(mean_x, mean_y), width=width, height=height,
                      angle=angle, facecolor=color, alpha=0.15, edgecolor=color, lw=2)
        ax.add_patch(ell)

ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.set_title('Connectivity Parameter Space\n(Prinz analog: solution space geometry)')
ax.legend()

# Panel C: PC1 loadings — what does "homogeneous scaling" look like?
ax = axes[1, 0]
loadings = pca.components_[0]
sorted_idx = np.argsort(np.abs(loadings))[::-1][:15]
short_names = [pca_feat_names[i].replace('_strength', '_str')
               .replace('RSU->RSU', 'R→R').replace('RSU->FS', 'R→F')
               .replace('FS->RSU', 'F→R').replace('FS->FS', 'F→F')
               .replace('_ei_ratio', '_ei')
               for i in sorted_idx]
colors = ['green' if loadings[i] > 0 else 'red' for i in sorted_idx]
ax.barh(range(len(sorted_idx)), [loadings[i] for i in sorted_idx], color=colors, alpha=0.7)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels(short_names, fontsize=7)
ax.set_xlabel('PC1 Loading')
ax.set_title('PC1 Loadings\n("Homogeneous scaling" direction?)')
ax.invert_yaxis()

# Panel D: Spread in PC space — do AD animals scatter more?
ax = axes[1, 1]
# Distance from group centroid for each animal
n_pcs = min(5, scores.shape[1])
wt_centroid = np.mean(scores[wt_mask, :n_pcs], axis=0)
ad_centroid = np.mean(scores[ad_mask, :n_pcs], axis=0)

wt_dists = np.linalg.norm(scores[wt_mask, :n_pcs] - wt_centroid, axis=1)
ad_dists = np.linalg.norm(scores[ad_mask, :n_pcs] - ad_centroid, axis=1)

parts = ax.violinplot([wt_dists, ad_dists], positions=[0, 1], showmeans=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(['#2196F3', '#F44336'][i])
    pc.set_alpha(0.7)

# Individual points
for i, (d, name) in enumerate(zip(wt_dists, np.array(animals)[wt_mask])):
    ax.scatter(0 + np.random.uniform(-0.05, 0.05), d, c='#2196F3', s=30, zorder=5)
for i, (d, name) in enumerate(zip(ad_dists, np.array(animals)[ad_mask])):
    ax.scatter(1 + np.random.uniform(-0.05, 0.05), d, c='#F44336', s=30, zorder=5)

stat, pval = stats.mannwhitneyu(wt_dists, ad_dists, alternative='two-sided')
ax.set_xticks([0, 1])
ax.set_xticklabels(['WT', 'AD'])
ax.set_ylabel(f'Distance from Group Centroid (first {n_pcs} PCs)')
ax.set_title(f'Spread in Parameter Space\nMann-Whitney p={pval:.4f}')

plt.tight_layout()
plt.show()

print(f'\nWT spread (mean dist from centroid): {np.mean(wt_dists):.2f} ± {np.std(wt_dists):.2f}')
print(f'AD spread (mean dist from centroid): {np.mean(ad_dists):.2f} ± {np.std(ad_dists):.2f}')

In [ ]:
# =============================================================================
# OUTLIER DETECTION 3: PCA — Per-Animal Distance & Leverage
# =============================================================================
# 1. Distance from global centroid in PC space (who sits far from everyone?)
# 2. Leverage: how much does removing each animal shift the PCA solution?
# 3. Leave-one-out: does the WT vs AD spread comparison change?
# =============================================================================

print('=' * 80)
print('OUTLIER DETECTION: PCA — Per-Animal Distance & Influence')
print('=' * 80)

n_pcs_check = min(5, scores.shape[1])

# Global centroid distance
global_centroid = np.mean(scores[:, :n_pcs_check], axis=0)
global_dists = np.linalg.norm(scores[:, :n_pcs_check] - global_centroid, axis=1)

# Per-genotype centroid distance (same as in original PCA cell)
wt_cent = np.mean(scores[wt_mask, :n_pcs_check], axis=0)
ad_cent = np.mean(scores[ad_mask, :n_pcs_check], axis=0)
dist_from_own = np.zeros(len(animals))
for i in range(len(animals)):
    cent = wt_cent if wt_mask[i] else ad_cent
    dist_from_own[i] = np.linalg.norm(scores[i, :n_pcs_check] - cent)

# Leave-one-out: how much does the spread comparison change?
# Original spread comparison
stat_orig, p_orig = stats.mannwhitneyu(wt_dists, ad_dists, alternative='two-sided')
spread_ratio_orig = np.mean(ad_dists) / np.mean(wt_dists) if np.mean(wt_dists) > 0 else 0

loo_spread_results = []
for i in range(len(animals)):
    # Remove animal i, recompute centroids and distances
    keep = np.ones(len(animals), dtype=bool)
    keep[i] = False
    
    wt_keep = wt_mask[keep]
    ad_keep = ad_mask[keep]
    scores_keep = scores[keep, :n_pcs_check]
    
    if wt_keep.sum() < 2 or ad_keep.sum() < 2:
        continue
    
    wt_c = np.mean(scores_keep[wt_keep], axis=0)
    ad_c = np.mean(scores_keep[ad_keep], axis=0)
    wt_d = np.linalg.norm(scores_keep[wt_keep] - wt_c, axis=1)
    ad_d = np.linalg.norm(scores_keep[ad_keep] - ad_c, axis=1)
    
    if len(wt_d) >= 2 and len(ad_d) >= 2:
        _, p_loo = stats.mannwhitneyu(wt_d, ad_d, alternative='two-sided')
    else:
        p_loo = np.nan
    
    ratio_loo = np.mean(ad_d) / np.mean(wt_d) if np.mean(wt_d) > 0 else 0
    loo_spread_results.append({
        'animal': animals[i], 'genotype': genotypes[i],
        'p_loo': p_loo, 'ratio_loo': ratio_loo,
        'delta_ratio': ratio_loo - spread_ratio_orig,
        'delta_p': p_loo - p_orig,
    })

# Print results
print(f'\nOriginal spread comparison: AD/WT ratio = {spread_ratio_orig:.3f}, p = {p_orig:.4f}')
print(f'\n{"Animal":<10s} {"Geno":>4s} {"GlobalDist":>10s} {"OwnDist":>10s} '
      f'{"LOO_ratio":>10s} {"Δratio":>10s} {"LOO_p":>10s} {"Δp":>10s}')
print('-' * 85)

for r in sorted(loo_spread_results, key=lambda x: abs(x['delta_ratio']), reverse=True):
    a = r['animal']
    i = animals.index(a)
    flag = ''
    if abs(r['delta_ratio']) > 0.15:
        flag = ' *** HIGH INFLUENCE'
        outlier_flags[a].append(f'PCA_SPREAD_INFLUENCE: Δratio={r["delta_ratio"]:+.3f}')
    print(f'{a:<10s} {r["genotype"]:>4s} {global_dists[i]:>10.3f} {dist_from_own[i]:>10.3f} '
          f'{r["ratio_loo"]:>10.3f} {r["delta_ratio"]:>+10.3f} '
          f'{r["p_loo"]:>10.4f} {r["delta_p"]:>+10.4f}{flag}')

# Visualization: PC1 vs PC2 with distance from centroid as size
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: PC space colored by global distance
ax = axes[0]
scatter_sizes = 50 + 200 * (global_dists / global_dists.max())
for i in range(len(animals)):
    color = '#F44336' if ad_mask[i] else '#2196F3'
    ax.scatter(scores[i, 0], scores[i, 1], s=scatter_sizes[i], c=color,
               alpha=0.7, edgecolors='black', linewidth=0.5, zorder=5)
    ax.annotate(animals[i], (scores[i, 0], scores[i, 1]), fontsize=6,
                xytext=(4, 4), textcoords='offset points')
ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.set_title('PC Space — Size = distance from global centroid\n(Larger = more extreme)',
             fontsize=11)

# Right: LOO influence on spread ratio
ax = axes[1]
loo_animals = [r['animal'] for r in sorted(loo_spread_results, 
               key=lambda x: x['delta_ratio'], reverse=True)]
loo_deltas = [r['delta_ratio'] for r in sorted(loo_spread_results,
              key=lambda x: x['delta_ratio'], reverse=True)]
loo_colors = ['#F44336' if r['genotype'] == 'AD' else '#2196F3' 
              for r in sorted(loo_spread_results, key=lambda x: x['delta_ratio'], reverse=True)]
ax.barh(range(len(loo_animals)), loo_deltas, color=loo_colors, 
        edgecolor='#333333', linewidth=0.5)
ax.axvline(0, color='black', linewidth=1)
ax.set_yticks(range(len(loo_animals)))
ax.set_yticklabels(loo_animals, fontsize=8)
ax.set_xlabel('Δ AD/WT spread ratio when removed')
ax.set_title('Leave-One-Out PCA Spread Influence\n(positive = removing increases AD/WT difference)',
             fontsize=11)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Cross-Region Coordination

**Schulz analog for electrically coupled neurons**: PD neurons in the same crab had similar  
channel expression (correlated within-animal). Analogously, if connectivity in AC is high  
for a given animal, is connectivity in V1 also high? This would suggest a global regulatory  
mechanism (like the activity-dependent feedback in Temporal et al. 2014).

If AD disrupts this cross-region coordination, it suggests the homeostatic feedback loop  
that normally keeps regions co-regulated has broken down.

In [ ]:
# For each animal, compute overall mean coupling strength per region
region_strengths = defaultdict(lambda: {'WT': [], 'AD': []})

for p in all_params:
    geno = p['genotype']
    for reg in REGIONS:
        key = f'{reg}_mean_strength'
        if key in p and p[key] > 0:
            region_strengths[reg][geno].append(p[key])

# Cross-region correlation for each genotype
# Build per-animal region strength vectors (only animals with all 4 regions present)
wt_reg_data = []
ad_reg_data = []
wt_names = []
ad_names = []

for p in all_params:
    vals = []
    has_all = True
    for reg in REGIONS:
        key = f'{reg}_mean_strength'
        if key in p and p[key] > 0:
            vals.append(p[key])
        else:
            has_all = False
            break
    if has_all:
        if p['genotype'] == 'WT':
            wt_reg_data.append(vals)
            wt_names.append(p['animal'])
        else:
            ad_reg_data.append(vals)
            ad_names.append(p['animal'])

wt_reg = np.array(wt_reg_data)
ad_reg = np.array(ad_reg_data)

print(f'Animals with all 4 regions (after MIN_NEURONS filter): {len(wt_reg)} WT, {len(ad_reg)} AD')

# Plot cross-region scatter (all pairs)
from itertools import combinations
reg_pairs = list(combinations(range(len(REGIONS)), 2))
n_pairs = len(reg_pairs)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flat

for pi, (ri, rj) in enumerate(reg_pairs):
    ax = axes[pi]
    scatter_with_regression(
        ax,
        wt_reg[:, ri], wt_reg[:, rj],
        ad_reg[:, ri], ad_reg[:, rj],
        f'{REGIONS[ri]} mean strength',
        f'{REGIONS[rj]} mean strength',
        f'{REGIONS[ri]} vs {REGIONS[rj]}'
    )

for i in range(n_pairs, len(axes.flat) if hasattr(axes, 'flat') else 6):
    axes[i].set_visible(False)

plt.suptitle('Cross-Region Coordination (Schulz PD neuron analog)\n'
             'WT: regions co-regulated? AD: coordination lost?', fontsize=13)
plt.tight_layout()
plt.show()

# Quantify with bootstrap CIs (correlations with small n are unreliable)
def bootstrap_cross_region_corr(reg_arr, n_boot=10000, seed=42):
    """Bootstrap mean cross-region correlation with 95% CI."""
    rng = np.random.RandomState(seed)
    n_animals = reg_arr.shape[0]
    n_regions = reg_arr.shape[1]
    
    # Observed
    corr_mat = np.corrcoef(reg_arr.T)
    observed = np.mean(corr_mat[np.triu_indices(n_regions, k=1)])
    
    # Bootstrap
    boot_means = []
    for _ in range(n_boot):
        idx = rng.choice(n_animals, n_animals, replace=True)
        boot_arr = reg_arr[idx]
        if any(np.std(boot_arr[:, c]) < 1e-10 for c in range(n_regions)):
            continue
        c = np.corrcoef(boot_arr.T)
        boot_means.append(np.mean(c[np.triu_indices(n_regions, k=1)]))
    
    boot_means = np.array(boot_means)
    ci_lo = np.percentile(boot_means, 2.5)
    ci_hi = np.percentile(boot_means, 97.5)
    return observed, ci_lo, ci_hi

if wt_reg.shape[0] >= 3:
    obs, lo, hi = bootstrap_cross_region_corr(wt_reg)
    print(f'\nWT mean cross-region r = {obs:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  (n={len(wt_reg)})')
if ad_reg.shape[0] >= 3:
    obs, lo, hi = bootstrap_cross_region_corr(ad_reg)
    print(f'AD mean cross-region r = {obs:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  (n={len(ad_reg)})')
print('\nNote: CIs are wide due to small n — interpret cross-region correlations with caution.')

In [ ]:
# =============================================================================
# OUTLIER DETECTION 4: Cross-Region Correlation — Leave-One-Out
# =============================================================================
# The cross-region coordination analysis is critical (feeds into the biphasic
# trajectory finding). Check how each animal affects the correlation.
# =============================================================================

print('=' * 80)
print('OUTLIER DETECTION: Cross-Region Coordination — Leave-One-Out')
print('=' * 80)

# Rebuild region strength arrays with animal names tracked
reg_all_data = []
reg_all_names = []
reg_all_genos = []

for p in all_params:
    vals = []
    has_all = True
    for reg in REGIONS:
        key = f'{reg}_mean_strength'
        if key in p and p[key] > 0:
            vals.append(p[key])
        else:
            has_all = False
            break
    if has_all:
        reg_all_data.append(vals)
        reg_all_names.append(p['animal'])
        reg_all_genos.append(p['genotype'])

reg_all_arr = np.array(reg_all_data)

def mean_cross_region_corr(arr):
    """Mean of upper triangle of correlation matrix."""
    if arr.shape[0] < 3:
        return np.nan
    c = np.corrcoef(arr.T)
    return np.mean(c[np.triu_indices(arr.shape[1], k=1)])

# Full correlations
wt_idx = [i for i, g in enumerate(reg_all_genos) if g == 'WT']
ad_idx = [i for i, g in enumerate(reg_all_genos) if g == 'AD']
r_wt_full = mean_cross_region_corr(reg_all_arr[wt_idx])
r_ad_full = mean_cross_region_corr(reg_all_arr[ad_idx])

print(f'\nFull dataset: WT cross-region r = {r_wt_full:.3f} (n={len(wt_idx)})')
print(f'              AD cross-region r = {r_ad_full:.3f} (n={len(ad_idx)})')

# Leave-one-out
print(f'\n{"Animal":<10s} {"Geno":>4s} {"LOO_r_group":>12s} {"Δr":>8s} {"Direction":>12s}')
print('-' * 55)

loo_region_results = []
for i, (name, geno) in enumerate(zip(reg_all_names, reg_all_genos)):
    if geno == 'WT':
        group_idx = [j for j in wt_idx if j != i]
        r_full = r_wt_full
    else:
        group_idx = [j for j in ad_idx if j != i]
        r_full = r_ad_full
    
    r_loo = mean_cross_region_corr(reg_all_arr[group_idx])
    delta = r_loo - r_full
    direction = '↑ stronger' if delta > 0 else '↓ weaker'
    
    loo_region_results.append({
        'animal': name, 'genotype': geno,
        'r_loo': r_loo, 'delta_r': delta
    })
    
    flag = ''
    if abs(delta) > 0.10:
        flag = ' *** HIGH INFLUENCE'
        outlier_flags[name].append(f'REGION_CORR_INFLUENCE: Δr={delta:+.3f}')
    
    print(f'{name:<10s} {geno:>4s} {r_loo:>12.3f} {delta:>+8.3f} {direction:>12s}{flag}')

# Visualization
fig, ax = plt.subplots(figsize=(12, 5))
sorted_results = sorted(loo_region_results, key=lambda x: x['delta_r'], reverse=True)
bar_animals = [r['animal'] for r in sorted_results]
bar_deltas = [r['delta_r'] for r in sorted_results]
bar_colors = ['#F44336' if r['genotype'] == 'AD' else '#2196F3' for r in sorted_results]

ax.bar(range(len(bar_animals)), bar_deltas, color=bar_colors,
       edgecolor='#333333', linewidth=0.5)
ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(range(len(bar_animals)))
ax.set_xticklabels(bar_animals, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Δr (cross-region corr change when removed)', fontsize=11)
ax.set_title('Cross-Region Coordination: Leave-One-Out Influence\n'
             '(positive = removing strengthens group correlation)',
             fontsize=13, fontweight='bold')
ax.legend(handles=[
    Line2D([0], [0], color='#2196F3', lw=8, label='WT'),
    Line2D([0], [0], color='#F44336', lw=8, label='AD'),
], fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

## 7. Solution Space Outlier Detection

**Prinz et al. analog**: Only ~20% of parameter combinations produce functional output.  
The "solution space" has boundaries. Animals outside these boundaries have dysfunctional  
network dynamics.

We define the WT "solution space" as the multivariate distribution of connectivity  
parameters across WT animals, then ask: how far are AD animals from this distribution?

- **Mahalanobis distance** from WT centroid (accounts for WT covariance structure)  
- AD animals far from the WT distribution are "outside the solution space"

In [ ]:
# Use PCA scores (avoids collinearity issues with Mahalanobis)
n_pcs_mahal = min(5, scores.shape[1], wt_mask.sum() - 1)  # need n > p
S = scores[:, :n_pcs_mahal]

# WT distribution
wt_mean = np.mean(S[wt_mask], axis=0)
wt_cov = np.cov(S[wt_mask].T)

# Regularize covariance if needed
wt_cov_reg = wt_cov + np.eye(n_pcs_mahal) * 1e-6
wt_cov_inv = np.linalg.inv(wt_cov_reg)

# Mahalanobis distance from WT centroid for ALL animals
mahal_dists = np.array([mahalanobis(S[i], wt_mean, wt_cov_inv) for i in range(len(animals))])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Mahalanobis distances
ax = axes[0]
wt_animals = np.array(animals)[wt_mask]
ad_animals = np.array(animals)[ad_mask]
wt_mahal = mahal_dists[wt_mask]
ad_mahal = mahal_dists[ad_mask]

# Sort each group
wt_sort = np.argsort(wt_mahal)
ad_sort = np.argsort(ad_mahal)

y_wt = np.arange(len(wt_mahal))
y_ad = np.arange(len(ad_mahal))

ax.barh(y_wt, wt_mahal[wt_sort], color='#2196F3', alpha=0.7, height=0.8)
ax.barh(y_ad + len(wt_mahal) + 1, ad_mahal[ad_sort], color='#F44336', alpha=0.7, height=0.8)

all_y = list(y_wt) + [len(wt_mahal)] + list(y_ad + len(wt_mahal) + 1)
all_labels = list(wt_animals[wt_sort]) + ['---'] + list(ad_animals[ad_sort])
ax.set_yticks(all_y)
ax.set_yticklabels(all_labels, fontsize=8)
ax.set_xlabel('Mahalanobis Distance from WT Centroid')
ax.set_title(f'Distance from WT "Solution Space"\n(first {n_pcs_mahal} PCs)')

# Chi2 threshold for outlier (p=0.05, dof=n_pcs)
chi2_thresh = np.sqrt(stats.chi2.ppf(0.95, df=n_pcs_mahal))
ax.axvline(chi2_thresh, color='red', ls='--', alpha=0.5, label=f'p=0.05 threshold ({chi2_thresh:.1f})')
ax.legend(fontsize=8)

# Panel B: Distribution comparison
ax = axes[1]
parts = ax.violinplot([wt_mahal, ad_mahal], positions=[0, 1], showmeans=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(['#2196F3', '#F44336'][i])
    pc.set_alpha(0.7)

for d in wt_mahal:
    ax.scatter(0 + np.random.uniform(-0.05, 0.05), d, c='#2196F3', s=40, zorder=5)
for d in ad_mahal:
    ax.scatter(1 + np.random.uniform(-0.05, 0.05), d, c='#F44336', s=40, zorder=5)

stat, pval = stats.mannwhitneyu(wt_mahal, ad_mahal, alternative='two-sided')
ax.set_xticks([0, 1])
ax.set_xticklabels(['WT', 'AD'])
ax.set_ylabel('Mahalanobis Distance from WT Centroid')
ax.set_title(f'Are AD animals outside the WT solution space?\nMann-Whitney p={pval:.4f}')
ax.axhline(chi2_thresh, color='red', ls='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f'\nWT Mahalanobis from WT centroid: {np.mean(wt_mahal):.2f} ± {np.std(wt_mahal):.2f}')
print(f'AD Mahalanobis from WT centroid: {np.mean(ad_mahal):.2f} ± {np.std(ad_mahal):.2f}')
print(f'\nOutliers (beyond p=0.05 threshold = {chi2_thresh:.2f}):')
for i in range(len(animals)):
    if mahal_dists[i] > chi2_thresh:
        print(f'  {animals[i]} ({genotypes[i]}): d={mahal_dists[i]:.2f}')

In [ ]:
# =============================================================================
# OUTLIER DETECTION 5: Mahalanobis — Per-Animal Influence on Solution Space
# =============================================================================
# The Mahalanobis distance defines who's "inside" vs "outside" the WT solution
# space. But with small n, removing one WT animal changes the centroid and
# covariance — potentially flipping who counts as an outlier.
# =============================================================================

print('=' * 80)
print('OUTLIER DETECTION: Mahalanobis — Leave-One-Out Solution Space Stability')
print('=' * 80)

# How many AD outliers are detected with the full WT reference?
n_ad_outliers_orig = np.sum(mahal_dists[ad_mask] > chi2_thresh)
n_wt_outliers_orig = np.sum(mahal_dists[wt_mask] > chi2_thresh)
print(f'\nFull WT reference: {n_ad_outliers_orig}/{ad_mask.sum()} AD outliers, '
      f'{n_wt_outliers_orig}/{wt_mask.sum()} WT outliers')
print(f'Chi2 threshold (p=0.05, {n_pcs_mahal} PCs): {chi2_thresh:.2f}')

# Leave-one-WT-out: how does removing each WT animal change the solution space?
print(f'\n--- Leave-One-WT-Out: Effect on AD outlier count ---')
print(f'{"Removed WT":<12s} {"AD outliers":>12s} {"Δ from orig":>12s} '
      f'{"Mean AD dist":>12s} {"Δ mean dist":>12s}')
print('-' * 65)

wt_idx_arr = np.where(wt_mask)[0]
ad_idx_arr = np.where(ad_mask)[0]

mean_ad_mahal_orig = np.mean(mahal_dists[ad_mask])

for k, wi in enumerate(wt_idx_arr):
    # Remove this WT animal, recompute reference
    keep_wt = np.delete(wt_idx_arr, k)
    S_wt_loo = scores[keep_wt, :n_pcs_mahal]
    
    if len(keep_wt) < n_pcs_mahal + 1:
        continue
    
    wt_mean_loo = np.mean(S_wt_loo, axis=0)
    wt_cov_loo = np.cov(S_wt_loo.T) + np.eye(n_pcs_mahal) * 1e-6
    wt_cov_inv_loo = np.linalg.inv(wt_cov_loo)
    
    # Recompute AD distances
    ad_mahal_loo = np.array([mahalanobis(scores[ai, :n_pcs_mahal], wt_mean_loo, wt_cov_inv_loo)
                             for ai in ad_idx_arr])
    n_ad_out_loo = np.sum(ad_mahal_loo > chi2_thresh)
    mean_ad_loo = np.mean(ad_mahal_loo)
    
    delta_n = n_ad_out_loo - n_ad_outliers_orig
    delta_mean = mean_ad_loo - mean_ad_mahal_orig
    
    flag = ''
    if abs(delta_n) >= 2 or abs(delta_mean) > 0.5:
        flag = ' *** HIGH INFLUENCE'
        outlier_flags[animals[wi]].append(
            f'MAHAL_INFLUENCE: removing shifts AD outlier count by {delta_n:+d}')
    
    print(f'{animals[wi]:<12s} {n_ad_out_loo:>12d} {delta_n:>+12d} '
          f'{mean_ad_loo:>12.2f} {delta_mean:>+12.2f}{flag}')

# Also check: which AD animal has the most extreme Mahalanobis distance?
print(f'\n--- AD Animals Ranked by Mahalanobis Distance ---')
print(f'{"Animal":<10s} {"Distance":>10s} {"p-value":>10s} {"Outlier?":>10s}')
print('-' * 45)
ad_names = np.array(animals)[ad_mask]
ad_m = mahal_dists[ad_mask]
for i in np.argsort(-ad_m):
    p_val = 1 - stats.chi2.cdf(ad_m[i]**2, df=n_pcs_mahal)
    is_out = 'YES' if ad_m[i] > chi2_thresh else 'no'
    print(f'{ad_names[i]:<10s} {ad_m[i]:>10.2f} {p_val:>10.4f} {is_out:>10s}')
    if ad_m[i] > chi2_thresh:
        outlier_flags[ad_names[i]].append(f'MAHAL_OUTLIER: d={ad_m[i]:.2f}, p={p_val:.4f}')

print(f'\n--- WT Animals Ranked by Mahalanobis Distance ---')
print(f'{"Animal":<10s} {"Distance":>10s} {"p-value":>10s} {"Outlier?":>10s}')
print('-' * 45)
wt_names = np.array(animals)[wt_mask]
wt_m = mahal_dists[wt_mask]
for i in np.argsort(-wt_m):
    p_val = 1 - stats.chi2.cdf(wt_m[i]**2, df=n_pcs_mahal)
    is_out = 'YES' if wt_m[i] > chi2_thresh else 'no'
    print(f'{wt_names[i]:<10s} {wt_m[i]:>10.2f} {p_val:>10.4f} {is_out:>10s}')
    if wt_m[i] > chi2_thresh:
        outlier_flags[wt_names[i]].append(f'MAHAL_OUTLIER_WT: d={wt_m[i]:.2f}, p={p_val:.4f}')

## 10. Age-Stratified Analysis

AD pathology is progressive — APP/PS1 mice develop amyloid plaques around 6-7 months.  
**Age cutoff: 212 days** (consistent with prior analyses in this dataset).

Two questions:
1. **Old AD vs Old WT**: Do aged AD mice show stronger effects than the overall comparison?
2. **Young AD vs Young WT**: Are effects already present pre-plaque, suggesting prodromal changes?

Groups:
- **Young WT** (5): BTH100, BTH90, BTH95, ZBR106, CAF42 (~73-207 days)
- **Young AD** (4): ZBR114, ZBR117, ZBR118, ZBR119 (~73-112 days) — ZBR113 excluded (insufficient neurons)
- **Old WT** (6): CAF69, CAF77, CAF81, CAF82, CAF88, CAF104 (~358-532 days)
- **Old AD** (4): CAF71, CAF73, CAF74, CAF75 (~381-492 days)

In [ ]:
# ── Age assignments (days) ──
ANIMAL_AGES = {
    'BTH100': 98.7, 'BTH90': 101.0, 'BTH95': 105.4,
    'CAF42': 207.4, 'CAF69': 358.2, 'CAF71': 492.3,
    'CAF73': 381.3, 'CAF74': 419.4, 'CAF75': 388.5,
    'CAF77': 394.3, 'CAF81': 455.2, 'CAF82': 455.2,
    'CAF88': 532.4, 'CAF104': 387.2,
    'ZBR106': 99.0, 'ZBR113': 91.0, 'ZBR114': 73.0,
    'ZBR117': 79.0, 'ZBR118': 99.0, 'ZBR119': 112.0,
}
AGE_CUTOFF = 212  # days

ages = np.array([ANIMAL_AGES[a] for a in animals])
young_mask = ages <= AGE_CUTOFF
old_mask = ages > AGE_CUTOFF

young_wt = young_mask & wt_mask
young_ad = young_mask & ad_mask
old_wt = old_mask & wt_mask
old_ad = old_mask & ad_mask

print(f'Young WT ({young_wt.sum()}): {[a for a, m in zip(animals, young_wt) if m]}')
print(f'Young AD ({young_ad.sum()}): {[a for a, m in zip(animals, young_ad) if m]}')
print(f'Old WT   ({old_wt.sum()}): {[a for a, m in zip(animals, old_wt) if m]}')
print(f'Old AD   ({old_ad.sum()}): {[a for a, m in zip(animals, old_ad) if m]}')

# ── 10a. PCA colored by age x genotype ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: PC1 vs PC2 colored by 4 groups
ax = axes[0]
group_info = [
    (young_wt, '#64B5F6', 'o', 'Young WT'),
    (young_ad, '#EF9A9A', 'o', 'Young AD'),
    (old_wt,   '#1565C0', 's', 'Old WT'),
    (old_ad,   '#B71C1C', 's', 'Old AD'),
]
for mask, color, marker, label in group_info:
    ax.scatter(scores[mask, 0], scores[mask, 1], c=color, s=100, alpha=0.85,
               marker=marker, label=label, edgecolors='black', linewidth=0.5, zorder=5)
for i, a in enumerate(animals):
    ax.annotate(a, (scores[i, 0], scores[i, 1]), fontsize=5.5,
                xytext=(4, 4), textcoords='offset points')
ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.set_title('Parameter Space by Age x Genotype')
ax.legend(fontsize=9)

# Right: Mahalanobis distance by 4 groups
ax = axes[1]
group_mahal = []
group_labels = []
group_colors = []
for mask, color, _, label in group_info:
    if mask.sum() > 0:
        group_mahal.append(mahal_dists[mask])
        group_labels.append(label)
        group_colors.append(color)

positions = range(len(group_mahal))
parts = ax.violinplot(group_mahal, positions=positions, showmeans=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(group_colors[i])
    pc.set_alpha(0.6)

for i, (vals, color) in enumerate(zip(group_mahal, group_colors)):
    for v in vals:
        ax.scatter(i + np.random.uniform(-0.06, 0.06), v, c=color, s=35, 
                   edgecolors='black', linewidth=0.3, zorder=5)

ax.axhline(chi2_thresh, color='red', ls='--', alpha=0.5, label=f'p=0.05 ({chi2_thresh:.1f})')
ax.set_xticks(positions)
ax.set_xticklabels(group_labels, fontsize=9)
ax.set_ylabel('Mahalanobis Distance from WT Centroid')
ax.set_title('Solution Space Distance by Age x Genotype')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ── 10b. Statistical comparisons ──
print('\n' + '=' * 70)
print('AGE-STRATIFIED ANALYSIS')
print('=' * 70)

# Mahalanobis comparisons
print('\n--- Mahalanobis Distance from WT Centroid ---')
for mask, label in [(young_wt, 'Young WT'), (young_ad, 'Young AD'),
                     (old_wt, 'Old WT'), (old_ad, 'Old AD')]:
    vals = mahal_dists[mask]
    n_out = np.sum(vals > chi2_thresh)
    print(f'  {label:10s}: {np.mean(vals):.2f} +/- {np.std(vals):.2f}  '
          f'(outliers: {n_out}/{mask.sum()})')

# Old AD vs Old WT
if old_ad.sum() >= 2 and old_wt.sum() >= 2:
    stat, p = stats.mannwhitneyu(mahal_dists[old_wt], mahal_dists[old_ad], alternative='two-sided')
    print(f'\n  Old AD vs Old WT: U={stat:.0f}, p={p:.4f}')
# Young AD vs Young WT
if young_ad.sum() >= 2 and young_wt.sum() >= 2:
    stat, p = stats.mannwhitneyu(mahal_dists[young_wt], mahal_dists[young_ad], alternative='two-sided')
    print(f'  Young AD vs Young WT: U={stat:.0f}, p={p:.4f}')
# Old AD vs Young AD (progression?)
if old_ad.sum() >= 2 and young_ad.sum() >= 2:
    stat, p = stats.mannwhitneyu(mahal_dists[young_ad], mahal_dists[old_ad], alternative='two-sided')
    print(f'  Old AD vs Young AD (progression?): U={stat:.0f}, p={p:.4f}')

# ── 10c. Key features by age x genotype ──
print('\n--- Key Connectivity Features (mean +/- std) ---')
region_features = [f'{r}_mean_strength' for r in REGIONS] + [f'{r}_density' for r in REGIONS]
region_features = [f for f in region_features if f in feature_names]

print(f'\n  {"Feature":30s} {"Young WT":>12s} {"Young AD":>12s} {"Old WT":>12s} {"Old AD":>12s}  old_p')
print('  ' + '-' * 95)
for f in region_features:
    idx = feature_names.index(f)
    ywt = X[young_wt, idx]
    yad = X[young_ad, idx]
    owt = X[old_wt, idx]
    oad = X[old_ad, idx]
    # Old AD vs Old WT test
    if old_ad.sum() >= 2 and old_wt.sum() >= 2:
        _, p_old = stats.mannwhitneyu(owt, oad, alternative='two-sided')
    else:
        p_old = np.nan
    marker = '*' if p_old < 0.05 else ''
    print(f'  {f:30s} {np.mean(ywt):.4f}+/-{np.std(ywt):.3f} '
          f'{np.mean(yad):.4f}+/-{np.std(yad):.3f} '
          f'{np.mean(owt):.4f}+/-{np.std(owt):.3f} '
          f'{np.mean(oad):.4f}+/-{np.std(oad):.3f}  {p_old:.3f} {marker}')

# ── 10d. Cell-type coupling by age x genotype (ANIMAL-LEVEL, no pseudoreplication) ──
print('\n--- Cell-Type Coupling Strength (averaged within animal, then compared) ---')
ct_pairs = ['RSU->RSU', 'RSU->FS', 'FS->RSU', 'FS->FS']

# First average across regions within each animal
animal_means = {pt: {'Young WT': [], 'Young AD': [], 'Old WT': [], 'Old AD': []} 
                for pt in ct_pairs}

for p in all_params:
    animal = p['animal']
    age = ANIMAL_AGES[animal]
    geno = p['genotype']
    age_label = 'Young' if age <= AGE_CUTOFF else 'Old'
    group = f'{age_label} {geno}'
    
    for pt in ct_pairs:
        # Collect this animal's values across all valid regions
        region_vals = []
        for reg in REGIONS:
            val = p.get(f'{reg}_{pt}_strength', 0)
            if val > 1e-6:
                region_vals.append(val)
        # Average across regions for this animal
        if region_vals:
            animal_means[pt][group].append(np.mean(region_vals))

print(f'\n  {"Pair":12s} {"Young WT":>12s} {"Young AD":>12s} {"Old WT":>12s} {"Old AD":>12s}  old_p   young_p')
print('  ' + '-' * 100)
for pt in ct_pairs:
    d = animal_means[pt]
    means = {g: np.mean(v) if v else 0 for g, v in d.items()}
    stds = {g: np.std(v) if v else 0 for g, v in d.items()}
    ns = {g: len(v) for g, v in d.items()}
    
    # Old comparison (animal-level, true n)
    if len(d['Old WT']) >= 2 and len(d['Old AD']) >= 2:
        _, p_old = stats.mannwhitneyu(d['Old WT'], d['Old AD'], alternative='two-sided')
    else:
        p_old = np.nan
    # Young comparison
    if len(d['Young WT']) >= 2 and len(d['Young AD']) >= 2:
        _, p_young = stats.mannwhitneyu(d['Young WT'], d['Young AD'], alternative='two-sided')
    else:
        p_young = np.nan
    
    m_old = '**' if p_old < 0.01 else '*' if p_old < 0.05 else ''
    m_young = '**' if p_young < 0.01 else '*' if p_young < 0.05 else ''
    
    print(f'  {pt:12s} {means["Young WT"]:.4f}+/-{stds["Young WT"]:.3f} '
          f'{means["Young AD"]:.4f}+/-{stds["Young AD"]:.3f} '
          f'{means["Old WT"]:.4f}+/-{stds["Old WT"]:.3f} '
          f'{means["Old AD"]:.4f}+/-{stds["Old AD"]:.3f}  '
          f'{p_old:.3f} {m_old:2s}  {p_young:.3f} {m_young}')

print(f'\n  (n per group: Young WT={ns["Young WT"]}, Young AD={ns["Young AD"]}, '
      f'Old WT={ns["Old WT"]}, Old AD={ns["Old AD"]})')

# ── 10e. Cross-region coordination by age (with bootstrap CIs) ──
print('\n--- Cross-Region Coordination with Bootstrap 95% CI ---')
for group_mask, label in [(young_wt, 'Young WT'), (young_ad, 'Young AD'),
                           (old_wt, 'Old WT'), (old_ad, 'Old AD')]:
    reg_data = []
    for pi, p in enumerate(all_params):
        if not group_mask[pi]:
            continue
        vals = []
        has_all = True
        for reg in REGIONS:
            key = f'{reg}_mean_strength'
            if key in p and p[key] > 0:
                vals.append(p[key])
            else:
                has_all = False
                break
        if has_all:
            reg_data.append(vals)
    
    if len(reg_data) >= 3:
        reg_arr = np.array(reg_data)
        obs, ci_lo, ci_hi = bootstrap_cross_region_corr(reg_arr)
        ci_width = ci_hi - ci_lo
        reliable = 'RELIABLE' if (ci_lo > 0 or ci_hi < 0) else 'CI spans zero'
        print(f'  {label:10s} (n={len(reg_data)}): r={obs:.3f}  '
              f'95% CI [{ci_lo:.3f}, {ci_hi:.3f}]  width={ci_width:.2f}  {reliable}')
    else:
        print(f'  {label:10s} (n={len(reg_data)}): too few animals for correlation')

print('\n  Caution: with n=4-5 per group, cross-region correlations are unstable.')
print('  Only trust results where the CI does not span zero.')

print('\n' + '=' * 70)

In [ ]:
# =============================================================================
# OUTLIER DETECTION 6: Age-Stratified Analysis — Per-Animal Influence
# =============================================================================
# The biphasic trajectory (Young AD: hypercoordination → Old AD: de-coordination)
# is the KEY finding. With only 4 Old AD animals and 4-5 Young AD animals,
# a single outlier could create or destroy this pattern.
# =============================================================================

print('=' * 80)
print('OUTLIER DETECTION: Age-Stratified — Biphasic Trajectory Robustness')
print('=' * 80)

# Rebuild age-stratified region data with names
age_group_reg = {'Young WT': [], 'Young AD': [], 'Old WT': [], 'Old AD': []}
age_group_names = {'Young WT': [], 'Young AD': [], 'Old WT': [], 'Old AD': []}

for pi, p in enumerate(all_params):
    a = p['animal']
    age = ANIMAL_AGES[a]
    geno = p['genotype']
    age_label = 'Young' if age <= AGE_CUTOFF else 'Old'
    group = f'{age_label} {geno}'
    
    vals = []
    has_all = True
    for reg in REGIONS:
        key = f'{reg}_mean_strength'
        if key in p and p[key] > 0:
            vals.append(p[key])
        else:
            has_all = False
            break
    if has_all:
        age_group_reg[group].append(vals)
        age_group_names[group].append(a)

# Compute original cross-region correlations per group
orig_corrs = {}
for group in age_group_reg:
    arr = np.array(age_group_reg[group])
    if arr.shape[0] >= 3:
        orig_corrs[group] = mean_cross_region_corr(arr)
    else:
        orig_corrs[group] = np.nan

print('\nOriginal cross-region correlations:')
for g in ['Young WT', 'Young AD', 'Old WT', 'Old AD']:
    n = len(age_group_reg[g])
    r = orig_corrs.get(g, np.nan)
    print(f'  {g:10s} (n={n}): r = {r:.3f}' if not np.isnan(r) else f'  {g:10s} (n={n}): too few')

# The biphasic trajectory metric: Young AD r minus Old AD r
# Large positive = hypercoordination → de-coordination pattern
if not np.isnan(orig_corrs.get('Young AD', np.nan)) and not np.isnan(orig_corrs.get('Old AD', np.nan)):
    trajectory_orig = orig_corrs['Young AD'] - orig_corrs['Old AD']
    print(f'\n  TRAJECTORY (Young AD r - Old AD r) = {trajectory_orig:.3f}')
    print(f'  (positive = young hypercoordinated relative to old)')
else:
    trajectory_orig = np.nan

# Leave-one-out for each group
print(f'\n{"="*70}')
print('Leave-One-Out: How does each animal affect its group\'s cross-region r?')
print(f'{"="*70}')

all_loo_age = []

for group in ['Young WT', 'Young AD', 'Old WT', 'Old AD']:
    arr = np.array(age_group_reg[group])
    names = age_group_names[group]
    r_orig = orig_corrs.get(group, np.nan)
    
    if len(names) < 4:
        print(f'\n{group}: n={len(names)}, skipping LOO (need >= 4)')
        continue
    
    print(f'\n--- {group} (n={len(names)}, original r={r_orig:.3f}) ---')
    print(f'  {"Removed":<10s} {"LOO_r":>8s} {"Δr":>8s} {"Direction":>12s}')
    print(f'  {"-"*42}')
    
    for i in range(len(names)):
        keep = np.ones(len(names), dtype=bool)
        keep[i] = False
        arr_loo = arr[keep]
        
        if arr_loo.shape[0] < 3:
            continue
        
        r_loo = mean_cross_region_corr(arr_loo)
        delta = r_loo - r_orig
        direction = '↑ stronger' if delta > 0 else '↓ weaker'
        
        flag = ''
        if abs(delta) > 0.15:
            flag = ' *** HIGH INFLUENCE'
            outlier_flags[names[i]].append(
                f'AGE_STRAT_INFLUENCE ({group}): Δr={delta:+.3f}')
        
        all_loo_age.append({
            'animal': names[i], 'group': group, 'r_orig': r_orig,
            'r_loo': r_loo, 'delta_r': delta
        })
        
        print(f'  {names[i]:<10s} {r_loo:>8.3f} {delta:>+8.3f} {direction:>12s}{flag}')

# Trajectory robustness: remove each AD animal and check if biphasic holds
if not np.isnan(trajectory_orig):
    print(f'\n{"="*70}')
    print('TRAJECTORY ROBUSTNESS: Young AD r - Old AD r')
    print(f'Original trajectory = {trajectory_orig:.3f}')
    print(f'{"="*70}')
    
    # Remove each Young AD animal
    print(f'\n--- Remove Young AD animals (one at a time) ---')
    print(f'  {"Removed":<10s} {"Young AD r":>12s} {"Trajectory":>12s} {"Δ":>8s} {"Holds?":>8s}')
    print(f'  {"-"*55}')
    
    ya_arr = np.array(age_group_reg['Young AD'])
    ya_names = age_group_names['Young AD']
    for i in range(len(ya_names)):
        keep = np.ones(len(ya_names), dtype=bool)
        keep[i] = False
        if keep.sum() < 3:
            continue
        r_ya_loo = mean_cross_region_corr(ya_arr[keep])
        traj_loo = r_ya_loo - orig_corrs['Old AD']
        delta = traj_loo - trajectory_orig
        holds = 'YES' if traj_loo > 0.2 else 'WEAK' if traj_loo > 0 else 'NO'
        print(f'  {ya_names[i]:<10s} {r_ya_loo:>12.3f} {traj_loo:>12.3f} {delta:>+8.3f} {holds:>8s}')
    
    # Remove each Old AD animal
    print(f'\n--- Remove Old AD animals (one at a time) ---')
    print(f'  {"Removed":<10s} {"Old AD r":>12s} {"Trajectory":>12s} {"Δ":>8s} {"Holds?":>8s}')
    print(f'  {"-"*55}')
    
    oa_arr = np.array(age_group_reg['Old AD'])
    oa_names = age_group_names['Old AD']
    for i in range(len(oa_names)):
        keep = np.ones(len(oa_names), dtype=bool)
        keep[i] = False
        if keep.sum() < 3:
            continue
        r_oa_loo = mean_cross_region_corr(oa_arr[keep])
        traj_loo = orig_corrs['Young AD'] - r_oa_loo
        delta = traj_loo - trajectory_orig
        holds = 'YES' if traj_loo > 0.2 else 'WEAK' if traj_loo > 0 else 'NO'
        print(f'  {oa_names[i]:<10s} {r_oa_loo:>12.3f} {traj_loo:>12.3f} {delta:>+8.3f} {holds:>8s}')

# Visualization: LOO influence on cross-region r, grouped by age x genotype
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
group_order = ['Young WT', 'Young AD', 'Old WT', 'Old AD']
group_colors_map = {'Young WT': '#64B5F6', 'Young AD': '#EF9A9A',
                    'Old WT': '#1565C0', 'Old AD': '#B71C1C'}

for gi, group in enumerate(group_order):
    ax = axes[gi // 2, gi % 2]
    group_results = [r for r in all_loo_age if r['group'] == group]
    if not group_results:
        ax.set_visible(False)
        continue
    
    group_results.sort(key=lambda x: x['delta_r'], reverse=True)
    names_g = [r['animal'] for r in group_results]
    deltas_g = [r['delta_r'] for r in group_results]
    
    ax.bar(range(len(names_g)), deltas_g, color=group_colors_map[group],
           edgecolor='#333333', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xticks(range(len(names_g)))
    ax.set_xticklabels(names_g, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Δr when removed', fontsize=10)
    r_orig_g = orig_corrs.get(group, np.nan)
    ax.set_title(f'{group} (n={len(names_g)}, r={r_orig_g:.3f})', fontsize=12, fontweight='bold')

plt.suptitle('Age-Stratified LOO: Cross-Region Coordination Influence',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# OUTLIER SUMMARY: All Flags Across All Analyses
# =============================================================================
# Tally up every outlier flag each animal received across all 6 detection
# analyses. Animals with many flags are consistently problematic.
# =============================================================================

print('=' * 80)
print('COMPREHENSIVE OUTLIER SUMMARY')
print('All flags accumulated across 6 outlier detection analyses')
print('=' * 80)

# Sort animals by total flag count
sorted_by_flags = sorted(outlier_flags.keys(), 
                         key=lambda a: len(outlier_flags[a]), reverse=True)

# Print full report
print(f'\n{"Animal":<10s} {"Geno":>4s} {"Age":>6s} {"#Flags":>7s}  Details')
print('-' * 90)
for a in sorted_by_flags:
    flags = outlier_flags[a]
    geno = genotypes[animals.index(a)] if a in animals else '?'
    age = ANIMAL_AGES.get(a, 0)
    n_flags = len(flags)
    
    if n_flags == 0:
        status = 'CLEAN'
    elif n_flags <= 2:
        status = 'MINOR'
    elif n_flags <= 4:
        status = 'MODERATE'
    else:
        status = '*** MAJOR ***'
    
    print(f'{a:<10s} {geno:>4s} {age:>5.0f}d {n_flags:>7d}  {status}')
    for f in flags:
        print(f'{"":>30s} - {f}')

# Visualization: flag count per animal
fig, ax = plt.subplots(figsize=(14, 6))
flag_counts = [len(outlier_flags[a]) for a in sorted_by_flags]
bar_colors = []
for a in sorted_by_flags:
    geno = genotypes[animals.index(a)] if a in animals else 'WT'
    bar_colors.append('#F44336' if geno == 'AD' else '#2196F3')

bars = ax.bar(range(len(sorted_by_flags)), flag_counts, color=bar_colors,
              edgecolor='#333333', linewidth=0.5)

# Color-code severity
for i, (bar, count) in enumerate(zip(bars, flag_counts)):
    if count >= 5:
        bar.set_edgecolor('red')
        bar.set_linewidth(2)

ax.set_xticks(range(len(sorted_by_flags)))
labels = [f'{a}\n({genotypes[animals.index(a)]}, {int(ANIMAL_AGES.get(a,0))}d)' 
          for a in sorted_by_flags]
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Number of outlier flags', fontsize=12)
ax.set_title('Comprehensive Animal Outlier Report\n'
             '(accumulated across Z-scores, correlations, PCA, Mahalanobis, '
             'cross-region, age-stratified)',
             fontsize=13, fontweight='bold')

# Threshold lines
if max(flag_counts) >= 3:
    ax.axhline(3, color='orange', ls='--', alpha=0.6, label='Moderate (3+)')
if max(flag_counts) >= 5:
    ax.axhline(5, color='red', ls='--', alpha=0.6, label='Major (5+)')

ax.legend(handles=[
    Line2D([0], [0], color='#2196F3', lw=8, label='WT'),
    Line2D([0], [0], color='#F44336', lw=8, label='AD'),
    Line2D([0], [0], color='orange', ls='--', label='Moderate threshold (3+)'),
    Line2D([0], [0], color='red', ls='--', label='Major threshold (5+)'),
], fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

# Cross-reference with spike classification findings
print(f'\n{"="*70}')
print('CROSS-REFERENCE WITH SPIKE CLASSIFICATION STABILITY TEST')
print(f'{"="*70}')
print(f'\nFrom xgboost_spike_classification.ipynb disruption scores:')
print(f'  zbr106 (+0.038) — most disruptive in spike classification')
print(f'  caf73  (+0.016) — second most disruptive')
print(f'  caf71  (-0.019) — strongest AD signal carrier')
print(f'  caf104 — excluded from spike classification (spike sorting instability)')
print(f'\nCheck if the same animals are flagged here ↑')

## 11. Strengthening the Coordination Trajectory

The key finding from Section 10 is a **biphasic trajectory** in cross-region coordination:
- **Young AD**: hyper-coordination (r ~ 0.7) — regions lock-step together
- **Old AD**: de-coordination (r ~ -0.3) — regions decouple
- **WT**: flat near zero at both ages

This mirrors the biphasic hyperexcitability trajectory in human AD (Dickerson & Sperling 2009):
early compensatory hyperactivity → late hypoactivity. Here we provide 5 additional analyses
to rigorously test this finding:

1. **Age as continuous variable** — avoid arbitrary cutoff, test genotype x age interaction
2. **Multiple connectivity metrics** — does the trajectory replicate across density, strength, E/I?
3. **Permutation test** — formal null hypothesis test for the trajectory
4. **Pairwise region correlations** — relax the all-4-regions requirement to include more animals
5. **Per-animal region profiles** — visualize the raw data driving the correlation

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11a. Age as Continuous Variable — Cross-Region Coordination vs Age
# ══════════════════════════════════════════════════════════════════════
# Instead of splitting at an arbitrary cutoff, compute each animal's
# cross-region coordination (mean pairwise correlation of region strengths)
# and plot it against age. Test genotype x age interaction.

def compute_animal_cross_region_corr(param_dict, regions=REGIONS):
    """Compute mean pairwise correlation of region strengths for one animal.
    
    Uses all available region pairs (doesn't require all 4 regions).
    Returns (mean_corr, n_regions_available).
    """
    strengths = {}
    for reg in regions:
        key = '{}_mean_strength'.format(reg)
        if key in param_dict and param_dict[key] > 0:
            strengths[reg] = param_dict[key]
    
    if len(strengths) < 2:
        return np.nan, len(strengths)
    
    # For a single animal with only aggregate strength per region,
    # we can't compute within-animal correlation (need multiple observations).
    # Instead, we return the coefficient of variation (inverse = coordination).
    # Low CV = regions have similar strength = coordinated.
    vals = list(strengths.values())
    mean_s = np.mean(vals)
    std_s = np.std(vals)
    # Use 1 - CV as a "coordination score" (1 = perfectly equal, 0 = very unequal)
    coordination = 1.0 - (std_s / mean_s) if mean_s > 1e-10 else np.nan
    return coordination, len(strengths)


# For the correlation-based measure, we need the GROUP-level approach:
# For each age x geno group, compute correlation across animals' region vectors.
# But we want a per-animal measure. So we use a leave-one-out approach:
# For each animal, compute how well its region profile matches the group pattern.

# Simpler approach: plot each animal's region-strength PROFILE as a vector,
# and compute its correlation with the GROUP MEAN profile of same-age peers.

def compute_profile_alignment(all_params_list, target_idx, same_group_indices, regions=REGIONS):
    """How well does animal's region profile align with group mean?"""
    target = all_params_list[target_idx]
    
    # Get target's region strengths
    target_vals = []
    valid_regs = []
    for reg in regions:
        key = '{}_mean_strength'.format(reg)
        if key in target and target[key] > 0:
            target_vals.append(target[key])
            valid_regs.append(reg)
    
    if len(valid_regs) < 3:
        return np.nan
    
    # Get group mean (excluding target) for same regions
    group_vals = []
    for reg in valid_regs:
        key = '{}_mean_strength'.format(reg)
        vals = []
        for gi in same_group_indices:
            if gi == target_idx:
                continue
            p = all_params_list[gi]
            if key in p and p[key] > 0:
                vals.append(p[key])
        if vals:
            group_vals.append(np.mean(vals))
        else:
            return np.nan
    
    if len(group_vals) < 3 or np.std(target_vals) < 1e-10 or np.std(group_vals) < 1e-10:
        return np.nan
    
    r, _ = stats.pearsonr(target_vals, group_vals)
    return r


# Compute per-animal coordination score (CV-based)
animal_ages_arr = []
animal_coord = []
animal_geno_color = []
animal_labels = []

for i, p in enumerate(all_params):
    age = ANIMAL_AGES[p['animal']]
    coord, n_reg = compute_animal_cross_region_corr(p)
    if not np.isnan(coord) and n_reg >= 3:
        animal_ages_arr.append(age)
        animal_coord.append(coord)
        animal_geno_color.append('#F44336' if p['genotype'] == 'AD' else '#2196F3')
        animal_labels.append(p['animal'])

animal_ages_arr = np.array(animal_ages_arr)
animal_coord = np.array(animal_coord)

# Separate by genotype
wt_idx = [i for i, c in enumerate(animal_geno_color) if c == '#2196F3']
ad_idx = [i for i, c in enumerate(animal_geno_color) if c == '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Coordination score (1 - CV) vs age
ax = axes[0]
for idx_list, color, label in [(wt_idx, '#2196F3', 'WT'), (ad_idx, '#F44336', 'AD')]:
    x = animal_ages_arr[idx_list]
    y = animal_coord[idx_list]
    ax.scatter(x, y, c=color, s=100, alpha=0.8, label=label,
               edgecolors='black', linewidth=0.5, zorder=5)
    for ii in idx_list:
        ax.annotate(animal_labels[ii], (animal_ages_arr[ii], animal_coord[ii]),
                    fontsize=6, xytext=(4, 4), textcoords='offset points')
    
    # Regression line
    if len(x) >= 3:
        slope, intercept, r, p_val, se = stats.linregress(x, y)
        xline = np.linspace(min(x), max(x), 50)
        ax.plot(xline, slope * xline + intercept, color=color, ls='--', lw=2)
        ax.text(0.02, 0.98 if color == '#2196F3' else 0.90,
                '{}: slope={:.4f}/day, r={:.2f}, p={:.3f}'.format(label, slope, r, p_val),
                transform=ax.transAxes, va='top', fontsize=9, color=color)

ax.axhline(0.5, color='gray', ls=':', alpha=0.3)
ax.set_xlabel('Age (days)')
ax.set_ylabel('Region Coordination Score (1 - CV)')
ax.set_title('Cross-Region Coordination vs Age\n'
             'Higher = regions have more similar connectivity strength')
ax.legend()
ax.axvline(AGE_CUTOFF, color='gray', ls='--', alpha=0.3, label='Age cutoff')

# Panel B: Same but using the GROUP cross-region correlation approach
# Compute rolling/sliding window cross-region correlation
ax = axes[1]

# For each AD animal, compute its cross-region corr with other AD animals nearby in age
# Use the 4-group approach but plot at actual ages
group_data = {
    'Young WT': {'age': [], 'corr': None, 'color': '#64B5F6'},
    'Young AD': {'age': [], 'corr': None, 'color': '#EF9A9A'},
    'Old WT':   {'age': [], 'corr': None, 'color': '#1565C0'},
    'Old AD':   {'age': [], 'corr': None, 'color': '#B71C1C'},
}

for pi, p in enumerate(all_params):
    age = ANIMAL_AGES[p['animal']]
    geno = p['genotype']
    age_label = 'Young' if age <= AGE_CUTOFF else 'Old'
    group = '{} {}'.format(age_label, geno)
    group_data[group]['age'].append(age)

# Recompute cross-region correlations per group (from Section 10e)
for group_mask, label in [(young_wt, 'Young WT'), (young_ad, 'Young AD'),
                           (old_wt, 'Old WT'), (old_ad, 'Old AD')]:
    reg_data = []
    for pi, p in enumerate(all_params):
        if not group_mask[pi]:
            continue
        vals = []
        has_all = True
        for reg in REGIONS:
            key = '{}_mean_strength'.format(reg)
            if key in p and p[key] > 0:
                vals.append(p[key])
            else:
                has_all = False
                break
        if has_all:
            reg_data.append(vals)
    
    if len(reg_data) >= 3:
        reg_arr = np.array(reg_data)
        obs, ci_lo, ci_hi = bootstrap_cross_region_corr(reg_arr)
        group_data[label]['corr'] = obs
        group_data[label]['ci'] = (ci_lo, ci_hi)

# Plot group-level cross-region correlations at mean age
for label, gd in group_data.items():
    if gd['corr'] is not None and gd['age']:
        mean_age = np.mean(gd['age'])
        ax.errorbar(mean_age, gd['corr'], 
                    yerr=[[gd['corr'] - gd['ci'][0]], [gd['ci'][1] - gd['corr']]],
                    fmt='o', color=gd['color'], markersize=12, capsize=8,
                    label='{} (r={:.2f})'.format(label, gd['corr']),
                    zorder=5, elinewidth=2)

# Connect AD points with arrow to show trajectory
ad_groups = ['Young AD', 'Old AD']
if all(group_data[g]['corr'] is not None and group_data[g]['age'] for g in ad_groups):
    x1 = np.mean(group_data['Young AD']['age'])
    y1 = group_data['Young AD']['corr']
    x2 = np.mean(group_data['Old AD']['age'])
    y2 = group_data['Old AD']['corr']
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='red', lw=2.5, ls='--'))
    ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.05, 'AD trajectory',
            ha='center', fontsize=10, color='red', fontweight='bold')

# Connect WT points
wt_groups = ['Young WT', 'Old WT']
if all(group_data[g]['corr'] is not None and group_data[g]['age'] for g in wt_groups):
    x1 = np.mean(group_data['Young WT']['age'])
    y1 = group_data['Young WT']['corr']
    x2 = np.mean(group_data['Old WT']['age'])
    y2 = group_data['Old WT']['corr']
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2.5, ls='--'))
    ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.05, 'WT trajectory',
            ha='center', fontsize=10, color='blue')

ax.axhline(0, color='gray', ls=':', alpha=0.5)
ax.set_xlabel('Age (days)')
ax.set_ylabel('Cross-Region Correlation (r)')
ax.set_title('Cross-Region Coordination Trajectory\n'
             'AD: hyper-coordination -> de-coordination')
ax.legend(fontsize=8, loc='lower left')
ax.set_ylim(-0.8, 1.0)

plt.tight_layout()
plt.show()

# Statistical test: genotype x age interaction on coordination score
print('\n=== Genotype x Age Interaction Test ===')
wt_ages = animal_ages_arr[wt_idx]
wt_coords = animal_coord[wt_idx]
ad_ages = animal_ages_arr[ad_idx]
ad_coords = animal_coord[ad_idx]

# Test: does the slope differ between genotypes?
wt_slope, wt_int, wt_r, wt_p, wt_se = stats.linregress(wt_ages, wt_coords)
ad_slope, ad_int, ad_r, ad_p, ad_se = stats.linregress(ad_ages, ad_coords)

print('  WT: slope = {:.5f}/day (r={:.3f}, p={:.3f})'.format(wt_slope, wt_r, wt_p))
print('  AD: slope = {:.5f}/day (r={:.3f}, p={:.3f})'.format(ad_slope, ad_r, ad_p))
print('  Slope difference: {:.5f}/day'.format(ad_slope - wt_slope))

# Interaction test via combined regression with dummy coding
all_ages_combined = np.concatenate([wt_ages, ad_ages])
all_coords_combined = np.concatenate([wt_coords, ad_coords])
geno_dummy = np.concatenate([np.zeros(len(wt_ages)), np.ones(len(ad_ages))])
interaction = geno_dummy * all_ages_combined

# Multiple regression: coord ~ age + geno + age*geno
X_reg = np.column_stack([all_ages_combined, geno_dummy, interaction, 
                         np.ones(len(all_ages_combined))])
# Least squares
beta, residuals, rank, sv = np.linalg.lstsq(X_reg, all_coords_combined, rcond=None)
y_pred = X_reg @ beta
ss_res = np.sum((all_coords_combined - y_pred) ** 2)
n_obs = len(all_coords_combined)
se_beta = np.sqrt(ss_res / (n_obs - 4) * np.diag(np.linalg.inv(X_reg.T @ X_reg)))
t_interaction = beta[2] / se_beta[2]
p_interaction = 2 * stats.t.sf(abs(t_interaction), df=n_obs - 4)

print('\n  Interaction term (age x genotype):')
print('    beta = {:.6f}, SE = {:.6f}'.format(beta[2], se_beta[2]))
print('    t = {:.3f}, p = {:.4f}'.format(t_interaction, p_interaction))
if p_interaction < 0.05:
    print('    ** SIGNIFICANT: AD coordination changes differently with age than WT **')
else:
    print('    Not significant at p<0.05 (but note small sample size)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11b. Multiple Connectivity Metrics — Same Trajectory?
# ══════════════════════════════════════════════════════════════════════
# If the trajectory is real, it should appear across multiple independent
# connectivity metrics: density, strength, E/I ratio, cell-type pairs.
# This is convergent validity — the strongest evidence against a fluke.

metrics_to_test = {
    'mean_strength': '{}_mean_strength',
    'density': '{}_density',
    'ei_ratio': '{}_ei_ratio',
    'RSU->RSU_strength': '{}_RSU->RSU_strength',
    'RSU->FS_strength': '{}_RSU->FS_strength',
    'FS->RSU_strength': '{}_FS->RSU_strength',
    'FS->FS_strength': '{}_FS->FS_strength',
    'RSU->RSU_density': '{}_RSU->RSU_density',
}

def compute_group_cross_region_corr_for_metric(metric_template, group_mask, 
                                                all_params_list, regions=REGIONS):
    """Compute cross-region correlation for a specific metric within a group."""
    reg_data = []
    for pi, p in enumerate(all_params_list):
        if not group_mask[pi]:
            continue
        vals = []
        has_all = True
        for reg in regions:
            key = metric_template.format(reg)
            if key in p and p[key] > 0:
                vals.append(p[key])
            else:
                has_all = False
                break
        if has_all:
            reg_data.append(vals)
    
    if len(reg_data) >= 3:
        reg_arr = np.array(reg_data)
        obs, ci_lo, ci_hi = bootstrap_cross_region_corr(reg_arr, n_boot=5000)
        return obs, ci_lo, ci_hi, len(reg_data)
    return None, None, None, len(reg_data)


fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flat

metric_results = {}

for mi, (metric_name, metric_template) in enumerate(metrics_to_test.items()):
    ax = axes[mi]
    
    results_this = {}
    for group_mask, label, color in [(young_wt, 'Young\nWT', '#64B5F6'),
                                      (young_ad, 'Young\nAD', '#EF9A9A'),
                                      (old_wt, 'Old\nWT', '#1565C0'),
                                      (old_ad, 'Old\nAD', '#B71C1C')]:
        obs, lo, hi, n = compute_group_cross_region_corr_for_metric(
            metric_template, group_mask, all_params, REGIONS)
        results_this[label] = (obs, lo, hi, n)
    
    metric_results[metric_name] = results_this
    
    # Plot bars with error bars
    x_pos = np.arange(4)
    labels_plot = list(results_this.keys())
    colors = ['#64B5F6', '#EF9A9A', '#1565C0', '#B71C1C']
    
    for xi, (lab, (obs, lo, hi, n)) in enumerate(results_this.items()):
        if obs is not None:
            yerr_lo = obs - lo
            yerr_hi = hi - obs
            ax.bar(xi, obs, color=colors[xi], alpha=0.7, edgecolor='black', linewidth=0.5)
            ax.errorbar(xi, obs, yerr=[[yerr_lo], [yerr_hi]], fmt='none',
                       color='black', capsize=5, elinewidth=1.5)
        else:
            ax.bar(xi, 0, color=colors[xi], alpha=0.3, edgecolor='black', linewidth=0.5)
            ax.text(xi, 0.02, 'n<3', ha='center', fontsize=7)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels_plot, fontsize=8)
    ax.axhline(0, color='gray', ls=':', alpha=0.5)
    ax.set_ylabel('Cross-Region r')
    ax.set_title(metric_name, fontsize=10, fontweight='bold')
    ax.set_ylim(-1.0, 1.0)

plt.suptitle('Cross-Region Coordination Across Multiple Metrics\n'
             'Convergent validity: does the trajectory replicate?',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print('\n=== Cross-Region Coordination by Metric ===')
print('{:25s} {:>15s} {:>15s} {:>15s} {:>15s}'.format(
    'Metric', 'Young WT', 'Young AD', 'Old WT', 'Old AD'))
print('-' * 90)

trajectory_count = 0
testable_count = 0
for metric_name, results in metric_results.items():
    row = '{:25s}'.format(metric_name)
    vals = {}
    for label, (obs, lo, hi, n) in results.items():
        if obs is not None:
            row += ' {:>7.3f} (n={})'.format(obs, n)
            vals[label] = obs
        else:
            row += ' {:>12s}'.format('n<3')
    print(row)
    
    # Only count metrics where both Young AD and Old AD are computable
    yad = vals.get('Young\nAD')
    oad = vals.get('Old\nAD')
    if yad is not None and oad is not None:
        testable_count += 1
        if yad > 0.3 and oad < 0.3 and yad > oad:
            trajectory_count += 1
            print('    ^ TRAJECTORY PATTERN: Young AD hyper -> Old AD de-coordination')
        else:
            print('    (testable but no trajectory pattern)')
    else:
        print('    (not testable: one or both AD groups have n<3)')

print('\nTrajectory pattern found in {}/{} TESTABLE metrics'.format(
    trajectory_count, testable_count))
print('({} metrics excluded: insufficient animals in one or both AD age groups)'.format(
    len(metrics_to_test) - testable_count))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11c. Permutation Test — Is the AD Trajectory Significant?
# ══════════════════════════════════════════════════════════════════════
# Null hypothesis: the young→old drop in cross-region coordination is
# no different in AD vs WT. We shuffle genotype labels 10,000 times and
# compute the trajectory statistic each time.
#
# Trajectory statistic = (Young_AD_corr - Old_AD_corr) - (Young_WT_corr - Old_WT_corr)
# This captures the INTERACTION: AD changes more than WT.

def compute_trajectory_stat(all_params_list, animals_list, genotypes_list, 
                            ages_arr, age_cutoff, regions=REGIONS):
    """Compute the trajectory interaction statistic.
    
    Returns: (young_ad_r - old_ad_r) - (young_wt_r - old_wt_r)
    Large positive = AD coordination drops much more than WT with age.
    """
    young = ages_arr <= age_cutoff
    old = ages_arr > age_cutoff
    geno_arr = np.array(genotypes_list)
    
    results = {}
    for age_mask, age_label in [(young, 'Young'), (old, 'Old')]:
        for geno in ['WT', 'AD']:
            mask = age_mask & (geno_arr == geno)
            reg_data = []
            for pi in np.where(mask)[0]:
                p = all_params_list[pi]
                vals = []
                has_all = True
                for reg in regions:
                    key = '{}_mean_strength'.format(reg)
                    if key in p and p[key] > 0:
                        vals.append(p[key])
                    else:
                        has_all = False
                        break
                if has_all:
                    reg_data.append(vals)
            
            if len(reg_data) >= 3:
                reg_arr = np.array(reg_data)
                corr_mat = np.corrcoef(reg_arr.T)
                r = np.mean(corr_mat[np.triu_indices(reg_arr.shape[1], k=1)])
                results['{} {}'.format(age_label, geno)] = r
            else:
                return np.nan  # can't compute
    
    if len(results) < 4:
        return np.nan
    
    ad_drop = results['Young AD'] - results['Old AD']
    wt_drop = results['Young WT'] - results['Old WT']
    return ad_drop - wt_drop


# Observed statistic
ages_for_perm = np.array([ANIMAL_AGES[a] for a in animals])
observed_stat = compute_trajectory_stat(all_params, animals, genotypes, 
                                         ages_for_perm, AGE_CUTOFF)
print('Observed trajectory statistic: {:.4f}'.format(observed_stat))
print('  (Young AD r - Old AD r) - (Young WT r - Old WT r)')
print('  Positive = AD coordination drops more with age than WT')

# Permutation test: shuffle genotype labels
n_perm = 10000
rng = np.random.RandomState(42)
perm_stats = []

for _ in range(n_perm):
    # Shuffle genotype labels (keep age structure intact)
    shuffled_genos = list(genotypes)
    rng.shuffle(shuffled_genos)
    
    stat = compute_trajectory_stat(all_params, animals, shuffled_genos,
                                    ages_for_perm, AGE_CUTOFF)
    if not np.isnan(stat):
        perm_stats.append(stat)

perm_stats = np.array(perm_stats)

# P-value: fraction of permutations with stat >= observed
p_perm = np.mean(perm_stats >= observed_stat)
p_two_sided = np.mean(np.abs(perm_stats) >= abs(observed_stat))

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(perm_stats, bins=50, color='gray', alpha=0.7, edgecolor='black', 
        linewidth=0.5, density=True)
ax.axvline(observed_stat, color='red', lw=3, ls='--', 
           label='Observed = {:.3f}'.format(observed_stat))
ax.axvline(0, color='black', lw=1, ls=':')

# Mark 95th percentile
p95 = np.percentile(perm_stats, 95)
ax.axvline(p95, color='orange', lw=2, ls=':', 
           label='95th percentile = {:.3f}'.format(p95))

ax.set_xlabel('Trajectory Statistic\n(AD drop - WT drop in cross-region coordination)')
ax.set_ylabel('Density')
ax.set_title('Permutation Test: Is the AD Trajectory Significant?\n'
             'p (one-sided) = {:.4f}, p (two-sided) = {:.4f}\n'
             '{:,} permutations, {:,} valid'.format(
                 p_perm, p_two_sided, n_perm, len(perm_stats)))
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print('\n=== Permutation Test Results ===')
print('  Observed trajectory stat: {:.4f}'.format(observed_stat))
print('  Null distribution: mean={:.4f}, std={:.4f}'.format(
    np.mean(perm_stats), np.std(perm_stats)))
print('  p-value (one-sided, AD drops more): {:.4f}'.format(p_perm))
print('  p-value (two-sided): {:.4f}'.format(p_two_sided))
print('  Valid permutations: {:,}/{:,}'.format(len(perm_stats), n_perm))
if p_perm < 0.05:
    print('\n  ** SIGNIFICANT: The AD coordination trajectory is unlikely by chance **')
    print('  ** AD mice lose cross-region coordination with age more than WT **')
else:
    print('\n  Not significant at p<0.05.')
    print('  Effect size (observed/null_std): {:.2f}'.format(
        observed_stat / np.std(perm_stats) if np.std(perm_stats) > 0 else 0))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11d. Pairwise Region Correlations (Relaxed Requirement)
# ══════════════════════════════════════════════════════════════════════
# The all-4-regions requirement excludes animals missing even 1 region.
# Here we compute correlations for EACH pair of regions independently,
# including all animals that have BOTH regions of the pair.
# This uses more data and tests whether the trajectory is region-general.

from itertools import combinations as combos

region_pair_names = list(combos(REGIONS, 2))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flat

pairwise_results = {}

for pi, (r1, r2) in enumerate(region_pair_names):
    ax = axes_flat[pi]
    
    # Collect strength values for both regions per animal
    pair_results = {}
    for group_mask, label, color in [(young_wt, 'Young WT', '#64B5F6'),
                                      (young_ad, 'Young AD', '#EF9A9A'),
                                      (old_wt, 'Old WT', '#1565C0'),
                                      (old_ad, 'Old AD', '#B71C1C')]:
        r1_vals = []
        r2_vals = []
        for i in np.where(group_mask)[0]:
            p = all_params[i]
            k1 = '{}_mean_strength'.format(r1)
            k2 = '{}_mean_strength'.format(r2)
            if k1 in p and p[k1] > 0 and k2 in p and p[k2] > 0:
                r1_vals.append(p[k1])
                r2_vals.append(p[k2])
        
        if len(r1_vals) >= 3:
            r, p_val = stats.pearsonr(r1_vals, r2_vals)
            pair_results[label] = {'r': r, 'p': p_val, 'n': len(r1_vals),
                                   'x': r1_vals, 'y': r2_vals}
        else:
            pair_results[label] = {'r': np.nan, 'p': np.nan, 'n': len(r1_vals),
                                   'x': r1_vals, 'y': r2_vals}
    
    pairwise_results['{}-{}'.format(r1, r2)] = pair_results
    
    # Plot scatter for all 4 groups
    for label, color, marker in [('Young WT', '#64B5F6', 'o'), 
                                  ('Young AD', '#EF9A9A', 'o'),
                                  ('Old WT', '#1565C0', 's'), 
                                  ('Old AD', '#B71C1C', 's')]:
        pr = pair_results[label]
        if pr['x']:
            ax.scatter(pr['x'], pr['y'], c=color, marker=marker, s=60, alpha=0.8,
                      edgecolors='black', linewidth=0.3,
                      label='{} r={:.2f}'.format(label, pr['r']) if not np.isnan(pr['r']) 
                            else '{} n={}'.format(label, pr['n']))
            # Add regression line if enough data
            if len(pr['x']) >= 3 and not np.isnan(pr['r']):
                slope, intercept = np.polyfit(pr['x'], pr['y'], 1)
                xline = np.linspace(min(pr['x']), max(pr['x']), 50)
                ax.plot(xline, slope * xline + intercept, color=color, ls='--', lw=1.5, alpha=0.7)
    
    ax.set_xlabel('{} mean strength'.format(r1))
    ax.set_ylabel('{} mean strength'.format(r2))
    ax.set_title('{} vs {}'.format(r1, r2), fontsize=11, fontweight='bold')
    ax.legend(fontsize=6, loc='upper left')

# Hide unused subplot
axes_flat[5].set_visible(False)

plt.suptitle('Pairwise Region Correlations (no all-4-regions requirement)\n'
             'Each subplot uses all animals with BOTH regions available',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print('\n=== Pairwise Region Correlation Summary ===')
print('{:12s} {:>20s} {:>20s} {:>20s} {:>20s}'.format(
    'Pair', 'Young WT', 'Young AD', 'Old WT', 'Old AD'))
print('-' * 95)

for pair_name, results in pairwise_results.items():
    row = '{:12s}'.format(pair_name)
    for label in ['Young WT', 'Young AD', 'Old WT', 'Old AD']:
        pr = results[label]
        if not np.isnan(pr['r']):
            row += ' r={:>5.2f} n={} p={:.2f}'.format(pr['r'], pr['n'], pr['p'])
        else:
            row += ' {:>18s}'.format('n={}'.format(pr['n']))
    print(row)

# Check trajectory across region pairs
print('\n--- Trajectory Pattern by Region Pair ---')
for pair_name, results in pairwise_results.items():
    yad = results['Young AD']['r'] if not np.isnan(results['Young AD']['r']) else None
    oad = results['Old AD']['r'] if not np.isnan(results['Old AD']['r']) else None
    if yad is not None and oad is not None:
        diff = yad - oad
        direction = 'HYPER->DE' if diff > 0.3 else 'stable' if abs(diff) < 0.3 else 'reverse'
        print('  {}: Young AD r={:.2f}, Old AD r={:.2f}, diff={:.2f}  [{}]'.format(
            pair_name, yad, oad, diff, direction))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11e. Per-Animal Region Strength Profiles
# ══════════════════════════════════════════════════════════════════════
# The most intuitive visualization: plot each animal's 4-region strength
# as a line (like a "radar" unwound to a line plot). If young AD animals
# have PARALLEL lines (coordinated), they'll look like scaled versions
# of each other. If old AD animals have CROSSING lines, their regions
# are independently varying.

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

group_masks_info = [
    (young_wt, 'Young WT', '#2196F3', axes[0, 0]),
    (young_ad, 'Young AD', '#F44336', axes[0, 1]),
    (old_wt, 'Old WT', '#1565C0', axes[1, 0]),
    (old_ad, 'Old AD', '#B71C1C', axes[1, 1]),
]

for group_mask, label, base_color, ax in group_masks_info:
    profiles = []
    names_in_group = []
    
    for i in np.where(group_mask)[0]:
        p = all_params[i]
        vals = []
        has_all = True
        for reg in REGIONS:
            key = '{}_mean_strength'.format(reg)
            if key in p and p[key] > 0:
                vals.append(p[key])
            else:
                has_all = False
                vals.append(0)
        
        if has_all:
            profiles.append(vals)
            names_in_group.append(p['animal'])
    
    if not profiles:
        ax.text(0.5, 0.5, 'No animals with all 4 regions', 
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label)
        continue
    
    profiles = np.array(profiles)
    
    # Plot each animal's profile
    x_pos = np.arange(len(REGIONS))
    cmap = plt.colormaps['Set2'] if 'WT' in label else plt.colormaps['Set1']
    
    for pi in range(len(profiles)):
        color = cmap(pi / max(len(profiles) - 1, 1))
        ax.plot(x_pos, profiles[pi], 'o-', color=color, lw=2.5, markersize=8,
                label=names_in_group[pi], alpha=0.8)
    
    # Add group mean
    mean_profile = np.mean(profiles, axis=0)
    ax.plot(x_pos, mean_profile, 'k--', lw=3, markersize=10, marker='D',
            label='Mean', alpha=0.7, zorder=10)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(REGIONS, fontsize=11)
    ax.set_ylabel('Mean Coupling Strength')
    
    # Compute profile similarity (mean pairwise correlation)
    if len(profiles) >= 2:
        pair_corrs = []
        for i in range(len(profiles)):
            for j in range(i + 1, len(profiles)):
                if np.std(profiles[i]) > 1e-10 and np.std(profiles[j]) > 1e-10:
                    r, _ = stats.pearsonr(profiles[i], profiles[j])
                    pair_corrs.append(r)
        
        if pair_corrs:
            mean_r = np.mean(pair_corrs)
            ax.set_title('{}\nMean pairwise r = {:.3f} (n={})'.format(
                label, mean_r, len(profiles)), fontsize=11)
        else:
            ax.set_title('{} (n={})'.format(label, len(profiles)))
    else:
        ax.set_title('{} (n={})'.format(label, len(profiles)))
    
    ax.legend(fontsize=7, loc='upper right')
    ax.set_ylim(bottom=0)

plt.suptitle('Per-Animal Region Strength Profiles\n'
             'Parallel lines = coordinated (high r), Crossing lines = de-coordinated (low r)\n'
             'Young AD should show parallel profiles; Old AD should show crossing profiles',
             fontsize=12, y=1.04)
plt.tight_layout()
plt.show()

# Also create a normalized version (z-scored within animal) to see SHAPE differences
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for group_mask, label, base_color, ax in [(young_wt, 'Young WT', '#2196F3', axes[0, 0]),
                                           (young_ad, 'Young AD', '#F44336', axes[0, 1]),
                                           (old_wt, 'Old WT', '#1565C0', axes[1, 0]),
                                           (old_ad, 'Old AD', '#B71C1C', axes[1, 1])]:
    profiles = []
    names_in_group = []
    
    for i in np.where(group_mask)[0]:
        p = all_params[i]
        vals = []
        has_all = True
        for reg in REGIONS:
            key = '{}_mean_strength'.format(reg)
            if key in p and p[key] > 0:
                vals.append(p[key])
            else:
                has_all = False
                vals.append(0)
        if has_all:
            profiles.append(vals)
            names_in_group.append(p['animal'])
    
    if not profiles:
        ax.text(0.5, 0.5, 'No animals with all 4 regions',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label)
        continue
    
    profiles = np.array(profiles)
    
    # Z-score within each animal (normalize to mean=0, std=1)
    # This removes absolute magnitude and shows SHAPE only
    profiles_z = np.zeros_like(profiles)
    for pi in range(len(profiles)):
        m = np.mean(profiles[pi])
        s = np.std(profiles[pi])
        if s > 1e-10:
            profiles_z[pi] = (profiles[pi] - m) / s
        else:
            profiles_z[pi] = 0
    
    x_pos = np.arange(len(REGIONS))
    cmap = plt.colormaps['Set2'] if 'WT' in label else plt.colormaps['Set1']
    
    for pi in range(len(profiles_z)):
        color = cmap(pi / max(len(profiles_z) - 1, 1))
        ax.plot(x_pos, profiles_z[pi], 'o-', color=color, lw=2.5, markersize=8,
                label=names_in_group[pi], alpha=0.8)
    
    ax.axhline(0, color='gray', ls=':', alpha=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(REGIONS, fontsize=11)
    ax.set_ylabel('Z-scored Strength (within animal)')
    ax.set_title('{} — Normalized Profiles\n'
                 'Similar shapes = coordinated regulation'.format(label), fontsize=10)
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Normalized Region Profiles (Shape Only, Magnitude Removed)\n'
             'Coordinated animals should have similar SHAPES across regions',
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

# Print summary statistics
print('\n=== Profile Similarity Summary ===')
print('Mean pairwise correlation of region profiles (parallel = coordinated):')
for group_mask, label in [(young_wt, 'Young WT'), (young_ad, 'Young AD'),
                           (old_wt, 'Old WT'), (old_ad, 'Old AD')]:
    profiles = []
    for i in np.where(group_mask)[0]:
        p = all_params[i]
        vals = []
        has_all = True
        for reg in REGIONS:
            key = '{}_mean_strength'.format(reg)
            if key in p and p[key] > 0:
                vals.append(p[key])
            else:
                has_all = False
                break
        if has_all:
            profiles.append(vals)
    
    if len(profiles) >= 2:
        pair_corrs = []
        for i in range(len(profiles)):
            for j in range(i + 1, len(profiles)):
                if np.std(profiles[i]) > 1e-10 and np.std(profiles[j]) > 1e-10:
                    r, _ = stats.pearsonr(profiles[i], profiles[j])
                    pair_corrs.append(r)
        if pair_corrs:
            print('  {:10s} (n={}): mean pairwise r = {:.3f} (range: {:.3f} to {:.3f})'.format(
                label, len(profiles), np.mean(pair_corrs), min(pair_corrs), max(pair_corrs)))
    else:
        print('  {:10s} (n={}): too few for correlation'.format(label, len(profiles)))

## Supplementary Analyses

The analyses below provide additional context for the degeneracy framework but did not
reveal strong AD vs WT differences when pooling all ages. They are retained here for
completeness.

- **Coefficient of Variation**: AD and WT have similar overall variability (CV ratio = 1.02x)
- **Within-Group Correlation Matrices**: Similar correlation structure when pooling ages
- **Conductance Ratio Analysis**: Most ratios show similar variance in AD vs WT
- **Summary Statistics**: Aggregated results from all analyses

## 2. Across-Animal Variability: Coefficient of Variation

Schulz et al. found 2-4x variability in channel expression across animals.  
If AD disrupts homeostatic regulation, we might see *increased* CV in AD animals —  
their parameters are less constrained, wandering further from functional set points.

Alternatively, AD might *reduce* variability if it collapses the network into a
narrower (but dysfunctional) regime.

In [ ]:
def compute_cv(values):
    """Coefficient of variation (std/mean), handling zero-mean."""
    m = np.mean(values)
    if abs(m) < 1e-10:
        return np.nan
    return np.std(values) / abs(m)

# CV per feature, split by genotype
# Focus on strength features (most analogous to conductances)
strength_features = [f for f in feature_names if '_strength' in f]
density_features = [f for f in feature_names if '_density' in f and 'RSU' in f or 'FS' in f]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: CV of coupling strengths
ax = axes[0]
wt_cvs = []
ad_cvs = []
labels = []
for f in strength_features:
    idx = feature_names.index(f)
    wt_cv = compute_cv(X[wt_mask, idx])
    ad_cv = compute_cv(X[ad_mask, idx])
    if not np.isnan(wt_cv) and not np.isnan(ad_cv):
        wt_cvs.append(wt_cv)
        ad_cvs.append(ad_cv)
        # Shorten label
        labels.append(f.replace('_strength', '').replace('_mean', ''))

x = np.arange(len(labels))
width = 0.35
ax.bar(x - width/2, wt_cvs, width, color='#2196F3', alpha=0.7, label='WT')
ax.bar(x + width/2, ad_cvs, width, color='#F44336', alpha=0.7, label='AD')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Coefficient of Variation')
ax.set_title('Variability in Coupling Strength Across Animals\n(Schulz et al. analog: 2-4x expected)')
ax.legend()
ax.axhline(1.0, color='gray', ls='--', alpha=0.3, label='CV=1')

# Panel B: Scatter — WT CV vs AD CV
ax = axes[1]
ax.scatter(wt_cvs, ad_cvs, s=60, c='purple', alpha=0.7)
lim = max(max(wt_cvs), max(ad_cvs)) * 1.1
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='Equal CV')
ax.set_xlabel('WT Coefficient of Variation')
ax.set_ylabel('AD Coefficient of Variation')
ax.set_title('Per-Feature Variability: AD vs WT\nAbove diagonal = AD more variable')
ax.legend()
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

# Statistical test: are CVs systematically different?
stat, pval = stats.wilcoxon(wt_cvs, ad_cvs)
print(f'\nWilcoxon signed-rank (paired CVs): stat={stat:.1f}, p={pval:.4f}')
print(f'Mean CV — WT: {np.mean(wt_cvs):.3f}, AD: {np.mean(ad_cvs):.3f}')
print(f'AD/WT CV ratio: {np.mean(ad_cvs)/np.mean(wt_cvs):.2f}x')

## 3. Within-Group Parameter Correlations

**The core Marder insight**: Individual parameters vary widely, but they *co-vary* —  
the ratios between conductances are what matter, not absolute values.  

Analogously, if RSU→FS coupling goes up in one animal, FS→RSU might also go up  
(maintaining E/I balance). If AD breaks this coordination, we'd see:
- WT: tight correlations between connection types (like PD neurons in same crab)
- AD: weaker or absent correlations (parameters vary independently)

We compute correlation matrices within each genotype group and compare.

In [ ]:
# Use strength features across regions — most analogous to conductances
# These are the "parameters" that should co-vary to maintain function
corr_features = [f for f in feature_names 
                 if ('_strength' in f or '_ei_ratio' in f or '_density' == f[-8:])  
                 and not f.endswith('_n_neurons')]

# Filter to features with nonzero variance in both groups
good_features = []
for f in corr_features:
    idx = feature_names.index(f)
    if np.std(X[wt_mask, idx]) > 1e-8 and np.std(X[ad_mask, idx]) > 1e-8:
        good_features.append(f)

print(f'Using {len(good_features)} features for correlation analysis')

feat_idx = [feature_names.index(f) for f in good_features]
X_wt = X[wt_mask][:, feat_idx]
X_ad = X[ad_mask][:, feat_idx]

# Correlation matrices
corr_wt = np.corrcoef(X_wt.T)
corr_ad = np.corrcoef(X_ad.T)

# Short labels for plotting
short_labels = [f.replace('_strength', '_str').replace('_density', '_den')
                .replace('_ei_ratio', '_ei').replace('RSU->RSU', 'R→R')
                .replace('RSU->FS', 'R→F').replace('FS->RSU', 'F→R')
                .replace('FS->FS', 'F→F') for f in good_features]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# WT correlation matrix
ax = axes[0]
im = ax.imshow(corr_wt, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=90, fontsize=5)
ax.set_yticks(range(len(short_labels)))
ax.set_yticklabels(short_labels, fontsize=5)
ax.set_title(f'WT Parameter Correlations (n={wt_mask.sum()})')
plt.colorbar(im, ax=ax, fraction=0.046)

# AD correlation matrix
ax = axes[1]
im = ax.imshow(corr_ad, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=90, fontsize=5)
ax.set_yticks(range(len(short_labels)))
ax.set_yticklabels(short_labels, fontsize=5)
ax.set_title(f'AD Parameter Correlations (n={ad_mask.sum()})')
plt.colorbar(im, ax=ax, fraction=0.046)

# Difference
ax = axes[2]
diff = corr_wt - corr_ad
vmax = np.percentile(np.abs(diff[~np.eye(len(diff), dtype=bool)]), 95)
im = ax.imshow(diff, cmap='PiYG', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=90, fontsize=5)
ax.set_yticks(range(len(short_labels)))
ax.set_yticklabels(short_labels, fontsize=5)
ax.set_title('Difference (WT - AD)\nGreen = WT more correlated')
plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

# Compare overall correlation strength
triu = np.triu_indices(len(good_features), k=1)
wt_corr_vals = np.abs(corr_wt[triu])
ad_corr_vals = np.abs(corr_ad[triu])

print(f'Mean |correlation| — WT: {np.mean(wt_corr_vals):.3f}, AD: {np.mean(ad_corr_vals):.3f}')
stat, pval = stats.mannwhitneyu(wt_corr_vals, ad_corr_vals, alternative='two-sided')
print(f'Mann-Whitney U (|corr| distributions): stat={stat:.0f}, p={pval:.4f}')

# Count strong correlations (|r| > 0.7)
wt_strong = np.mean(wt_corr_vals > 0.7)
ad_strong = np.mean(ad_corr_vals > 0.7)
print(f'Fraction |r| > 0.7 — WT: {wt_strong:.1%}, AD: {ad_strong:.1%}')

## 8. Conductance Ratio Analysis (Marder Framework)

The Marder lab showed that absolute conductance values matter less than **ratios**.  
Two neurons with 2x different gNa can fire identically if gK also scales 2x.  

For connectivity, the analog is: the ratio of RSU→FS to FS→RSU coupling matters more  
than either alone. If these ratios are tightly regulated in WT but variable in AD,  
that's evidence for loss of the homeostatic coordination.

In [ ]:
# Compute key ratios per animal per region
ratio_data = defaultdict(lambda: {'WT': [], 'AD': [], 'WT_names': [], 'AD_names': []})

for p in all_params:
    geno = p['genotype']
    animal = p['animal']
    
    for reg in REGIONS:
        # Ratio 1: RSU→FS / FS→RSU (reciprocal balance)
        rf = p.get(f'{reg}_RSU->FS_strength', 0)
        fr = p.get(f'{reg}_FS->RSU_strength', 0)
        if rf > 1e-6 and fr > 1e-6:
            ratio = np.log2(rf / fr)  # log ratio centered at 0
            ratio_data[f'{reg}_RF/FR'][geno].append(ratio)
            ratio_data[f'{reg}_RF/FR'][f'{geno}_names'].append(animal)
        
        # Ratio 2: RSU→RSU / FS→FS (principal pair balance)
        rr = p.get(f'{reg}_RSU->RSU_strength', 0)
        ff = p.get(f'{reg}_FS->FS_strength', 0)
        if rr > 1e-6 and ff > 1e-6:
            ratio = np.log2(rr / ff)
            ratio_data[f'{reg}_RR/FF'][geno].append(ratio)
            ratio_data[f'{reg}_RR/FF'][f'{geno}_names'].append(animal)
        
        # Ratio 3: Exc / Inh overall
        exc_key = f'{reg}_RSU->RSU_exc'
        inh_key = f'{reg}_RSU->RSU_inh'
        e = p.get(exc_key, 0)
        i_val = p.get(inh_key, 0)
        if e > 1e-6 and i_val > 1e-6:
            ratio = np.log2(e / i_val)
            ratio_data[f'{reg}_E/I'][geno].append(ratio)
            ratio_data[f'{reg}_E/I'][f'{geno}_names'].append(animal)

# Plot ratio distributions
ratio_keys = sorted(ratio_data.keys())
# Filter to ratios with enough data
ratio_keys = [k for k in ratio_keys 
              if len(ratio_data[k]['WT']) >= 3 and len(ratio_data[k]['AD']) >= 3]

n_ratios = len(ratio_keys)
ncols = 4
nrows = (n_ratios + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
if nrows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flat

for i, key in enumerate(ratio_keys):
    ax = axes[i]
    wt_vals = ratio_data[key]['WT']
    ad_vals = ratio_data[key]['AD']
    
    parts = ax.violinplot([wt_vals, ad_vals], positions=[0, 1], 
                          showmeans=True, showmedians=True)
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor(['#2196F3', '#F44336'][j])
        pc.set_alpha(0.7)
    
    # Individual points
    for v in wt_vals:
        ax.scatter(0 + np.random.uniform(-0.08, 0.08), v, c='#2196F3', s=25, zorder=5)
    for v in ad_vals:
        ax.scatter(1 + np.random.uniform(-0.08, 0.08), v, c='#F44336', s=25, zorder=5)
    
    ax.axhline(0, color='gray', ls='--', alpha=0.3)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['WT', 'AD'])
    ax.set_ylabel('log2(ratio)')
    
    # Compare variance (Levene's test)
    stat_l, p_levene = stats.levene(wt_vals, ad_vals)
    # Compare mean (Mann-Whitney)
    stat_m, p_mw = stats.mannwhitneyu(wt_vals, ad_vals, alternative='two-sided')
    
    ax.set_title(f'{key}\nvar p={p_levene:.3f}, loc p={p_mw:.3f}', fontsize=9)

for i in range(n_ratios, nrows * ncols):
    axes[i].set_visible(False)

plt.suptitle('Parameter Ratios: AD vs WT (Marder Framework)\n'
             'Key question: Is AD *variance* different? (Levene\'s test)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\n=== Ratio Variance Comparison (Levene\'s test) ===')
print('Significant variance difference = AD lost ratio regulation')
for key in ratio_keys:
    wt_vals = ratio_data[key]['WT']
    ad_vals = ratio_data[key]['AD']
    stat_l, p_l = stats.levene(wt_vals, ad_vals)
    wt_std = np.std(wt_vals)
    ad_std = np.std(ad_vals)
    marker = '***' if p_l < 0.001 else '**' if p_l < 0.01 else '*' if p_l < 0.05 else ''
    print(f'  {key:20s}: WT std={wt_std:.3f}, AD std={ad_std:.3f}, '
          f'ratio={ad_std/wt_std:.2f}x, p={p_l:.4f} {marker}')

## 9. Summary Statistics

In [ ]:
print('=' * 70)
print('DEGENERACY ANALYSIS SUMMARY')
print('=' * 70)
print()
print('Framework: Marder lab (Prinz 2004, Schulz 2006, Temporal 2014, Goaillard & Marder 2021)')
print('Hypothesis: AD disrupts covariance structure, not just mean connectivity')
print()

# 1. Variability
print('--- 1. Across-Animal Variability (Schulz analog) ---')
wt_mean_cv = np.mean(wt_cvs)
ad_mean_cv = np.mean(ad_cvs)
print(f'  Mean CV of coupling strengths: WT={wt_mean_cv:.3f}, AD={ad_mean_cv:.3f}')
print(f'  AD/WT CV ratio: {ad_mean_cv/wt_mean_cv:.2f}x')
stat, pval = stats.wilcoxon(wt_cvs, ad_cvs)
print(f'  Wilcoxon paired test: p={pval:.4f}')
print()

# 2. Correlation structure
print('--- 2. Parameter Correlation Structure ---')
print(f'  Mean |correlation| — WT: {np.mean(wt_corr_vals):.3f}, AD: {np.mean(ad_corr_vals):.3f}')
print(f'  Fraction strong correlations (|r|>0.7) — WT: {wt_strong:.1%}, AD: {ad_strong:.1%}')
print()

# 3. PCA
print('--- 3. Solution Space (PCA) ---')
print(f'  PCs for 80% variance: {n_80}')
print(f'  WT spread: {np.mean(wt_dists):.2f} ± {np.std(wt_dists):.2f}')
print(f'  AD spread: {np.mean(ad_dists):.2f} ± {np.std(ad_dists):.2f}')
print()

# 4. Solution space membership
print('--- 4. Solution Space Membership (Mahalanobis) ---')
print(f'  Mean Mahalanobis from WT centroid — WT: {np.mean(wt_mahal):.2f}, AD: {np.mean(ad_mahal):.2f}')
n_wt_outlier = np.sum(wt_mahal > chi2_thresh)
n_ad_outlier = np.sum(ad_mahal > chi2_thresh)
print(f'  Outliers (p<0.05) — WT: {n_wt_outlier}/{wt_mask.sum()}, AD: {n_ad_outlier}/{ad_mask.sum()}')
print()

# 5. Ratio regulation
print('--- 5. Ratio Regulation (Marder Framework) ---')
sig_ratios = []
for key in ratio_keys:
    wt_vals = ratio_data[key]['WT']
    ad_vals = ratio_data[key]['AD']
    stat_l, p_l = stats.levene(wt_vals, ad_vals)
    if p_l < 0.05:
        direction = 'AD more variable' if np.std(ad_vals) > np.std(wt_vals) else 'AD less variable'
        sig_ratios.append((key, p_l, direction))
        print(f'  {key}: p={p_l:.4f} ({direction})')

if not sig_ratios:
    print('  No significant differences in ratio variance (p<0.05)')

print()
print('=' * 70)